In [ ]:
%load_ext autoreload
%autoreload 0

# Flight Delay ML Platform — Historical Weather Forecasts

## 1. Weather Data Feasibility

The previous experiments demonstrated limited incremental
predictive value from historical delay statistics,
scheduled traffic, and alternative boosting models.

This experiment investigates external weather forecasts
as an additional source of predictive information.

### Proposed Data Source

NOAA High-Resolution Rapid Refresh (HRRR).

### Prediction Contract

- Prediction horizon: 24 hours before scheduled departure
- Weather source: archived numerical weather forecasts
- Forecast initialization and valid times are preserved
- Assumed forecast publication lag: 3 hours
- No future observed weather or reanalysis data
- Final 2025 test labels remain excluded

### Initial Scope

Six origin airports: ATL, DFW, DEN, ORD, LAX, SFO.

The initial pilot verifies archive coverage and variable
availability before large-scale feature generation.

In [ ]:
import re
import time
from pathlib import Path
from urllib.request import urlopen, Request

import numpy as np
import pandas as pd

ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

assert (ROOT / "pyproject.toml").is_file()

HRRR_BASE_URL = (
    "https://noaa-hrrr-bdp-pds.s3.amazonaws.com"
)

WEATHER_AVAILABILITY_LAG_HOURS = 3
WEATHER_MODEL = "hrrr"
WEATHER_PRODUCT = "wrfsfc"

ORIGIN_AIRPORTS = [
    "ATL", "DFW", "DEN", "ORD", "LAX", "SFO"
]

print("Project root:", ROOT)
print("Weather model:", WEATHER_MODEL)
print("Weather product:", WEATHER_PRODUCT)
print("Availability lag:", WEATHER_AVAILABILITY_LAG_HOURS)

In [ ]:
def build_hrrr_reference(run_at, forecast_hour):

    run_at = pd.Timestamp(run_at)

    if run_at.tzinfo is None:
        raise ValueError("UTC timestamp required")

    run_at = run_at.tz_convert("UTC")

    if run_at.hour not in [0, 6, 12, 18]:
        raise ValueError("Extended HRRR cycle required")

    if run_at.minute != 0 or run_at.second != 0:
        raise ValueError("Invalid initialization time")

    if not isinstance(forecast_hour, int):
        raise ValueError("Integer forecast hour required")

    if not 0 <= forecast_hour <= 48:
        raise ValueError("Invalid forecast horizon")

    date = run_at.strftime("%Y%m%d")
    cycle = run_at.strftime("%H")

    filename = (
        f"hrrr.t{cycle}z."
        f"wrfsfcf{forecast_hour:02d}.grib2"
    )

    url = (
        f"{HRRR_BASE_URL}/"
        f"hrrr.{date}/conus/{filename}"
    )

    return {
        "forecast_run_at_utc": run_at,
        "assumed_available_at_utc": (
            run_at + pd.Timedelta(
                hours=WEATHER_AVAILABILITY_LAG_HOURS
            )
        ),
        "forecast_valid_at_utc": (
            run_at + pd.Timedelta(hours=forecast_hour)
        ),
        "forecast_hour": forecast_hour,
        "grib_url": url,
        "index_url": url + ".idx",
    }


example = build_hrrr_reference(
    "2024-07-14T06:00:00Z",
    30,
)

display(pd.Series(example))

In [ ]:
PILOT_RUNS = [
    ("2022-12-30T18:00:00Z", 30),
    ("2023-01-15T06:00:00Z", 30),
    ("2023-07-14T06:00:00Z", 30),
    ("2023-12-15T06:00:00Z", 30),
    ("2024-01-15T06:00:00Z", 30),
    ("2024-07-14T06:00:00Z", 30),
]

archive_records = []
inventory_by_run = {}

for run_at, forecast_hour in PILOT_RUNS:

    reference = build_hrrr_reference(
        run_at,
        forecast_hour,
    )

    record = {
        "RunAtUTC": reference["forecast_run_at_utc"],
        "ForecastHour": forecast_hour,
        "ValidAtUTC": reference["forecast_valid_at_utc"],
        "IndexAvailable": False,
        "GRIBAvailable": False,
        "GRIBSizeMB": np.nan,
        "InventoryRecords": 0,
        "Error": "",
    }

    try:
        with urlopen(
            reference["index_url"],
            timeout=30,
        ) as response:

            inventory = response.read().decode("utf-8")

        lines = [
            line for line in inventory.splitlines()
            if line.strip()
        ]

        record["IndexAvailable"] = True
        record["InventoryRecords"] = len(lines)

        inventory_by_run[run_at] = lines

        request = Request(
            reference["grib_url"],
            method="HEAD",
        )

        with urlopen(request, timeout=30) as response:

            record["GRIBAvailable"] = (
                response.status == 200
            )

            size = response.headers.get("Content-Length")

            if size is not None:
                record["GRIBSizeMB"] = (
                    int(size) / 1024**2
                )

    except Exception as exc:

        record["Error"] = str(exc)

    archive_records.append(record)

    print(record)

archive_audit = pd.DataFrame(archive_records)

display(archive_audit)

print("\n=== Archive Availability Summary ===")

print(
    "Available indexes:",
    archive_audit["IndexAvailable"].sum(),
    "/",
    len(archive_audit),
)

print(
    "Available GRIB files:",
    archive_audit["GRIBAvailable"].sum(),
    "/",
    len(archive_audit),
)

In [ ]:
VARIABLE_PATTERNS = {
    "Temperature": r":TMP:2 m above ground:",
    "UWind": r":UGRD:10 m above ground:",
    "VWind": r":VGRD:10 m above ground:",
    "WindGust": r":GUST:surface:",
    "Visibility": r":VIS:surface:",
    "Precipitation": r":APCP:surface:",
    "CAPE": r":CAPE:",
    "CompositeReflectivity": r":REFC:",
    "CloudCeiling": r":HGT:cloud ceiling:",
}

variable_records = []

for run_at, lines in inventory_by_run.items():

    for variable, pattern in VARIABLE_PATTERNS.items():

        matches = [
            line for line in lines
            if re.search(pattern, line)
        ]

        variable_records.append({
            "RunAtUTC": run_at,
            "Variable": variable,
            "Available": len(matches) > 0,
            "MatchingMessages": len(matches),
            "Example": matches[0] if matches else "",
        })

variable_audit = pd.DataFrame(variable_records)

if not variable_audit.empty:

    display(
        variable_audit.pivot(
            index="RunAtUTC",
            columns="Variable",
            values="Available",
        )
    )

    display(
        variable_audit[
            ["Variable", "MatchingMessages", "Example"]
        ].drop_duplicates("Variable")
    )

else:
    print("No forecast inventories available.")

In [ ]:
if inventory_by_run:

    example_run = next(iter(inventory_by_run))

    print("Example run:", example_run)

    for line in inventory_by_run[example_run]:
        print(line)

## 2. Airport Weather Extraction

Extract historical numerical weather forecasts at six
origin airport locations using nearest-grid-point sampling.

The pilot uses HTTP byte-range requests to retrieve
selected GRIB2 messages instead of downloading complete
forecast files.

Forecast initialization time, valid time, forecast lead,
assumed availability time, and spatial sampling distance
are retained for provenance and quality control.

In [ ]:
import eccodes

AIRPORT_COORDINATES = pd.DataFrame([
    ("ATL", 33.6367, -84.4281),
    ("DFW", 32.8968, -97.0380),
    ("DEN", 39.8561, -104.6737),
    ("ORD", 41.9742, -87.9073),
    ("LAX", 33.9425, -118.4081),
    ("SFO", 37.6213, -122.3790),
], columns=[
    "Origin",
    "Latitude",
    "Longitude",
])

assert AIRPORT_COORDINATES["Origin"].is_unique

assert set(AIRPORT_COORDINATES["Origin"]) == set(
    ORIGIN_AIRPORTS
)

display(AIRPORT_COORDINATES)

In [ ]:
def select_hrrr_forecast(scheduled_departure_utc):

    departure = pd.Timestamp(
        scheduled_departure_utc
    )

    if pd.isna(departure):
        raise ValueError("Unresolved departure timestamp")

    if departure.tzinfo is None:
        raise ValueError("Timezone-aware timestamp required")

    departure = departure.tz_convert("UTC")

    prediction_at = (
        departure - pd.Timedelta(hours=24)
    )

    lag = pd.Timedelta(
        hours=WEATHER_AVAILABILITY_LAG_HOURS
    )

    run_at = (
        prediction_at - lag
    ).floor("6h")

    valid_at = departure.floor("h")

    forecast_hour = int(
        (valid_at - run_at)
        / pd.Timedelta(hours=1)
    )

    reference = build_hrrr_reference(
        run_at,
        forecast_hour,
    )

    reference["prediction_at_utc"] = prediction_at
    reference["scheduled_departure_utc"] = departure

    assert (
        reference["assumed_available_at_utc"]
        <= prediction_at
    )

    assert (
        reference["forecast_valid_at_utc"]
        == valid_at
    )

    assert 27 <= forecast_hour <= 32

    assert pd.Timedelta(0) <= (
        departure - valid_at
    ) < pd.Timedelta(hours=1)

    return reference

In [ ]:
examples = [
    "2024-07-15T08:59:59Z",
    "2024-07-15T09:00:00Z",
    "2024-07-15T09:00:01Z",
    "2024-07-15T12:00:00Z",
    "2024-07-15T14:35:00Z",
    "2023-01-01T07:20:00Z",
]

forecast_selection = pd.DataFrame([
    select_hrrr_forecast(value)
    for value in examples
])

display(
    forecast_selection[[
        "scheduled_departure_utc",
        "prediction_at_utc",
        "forecast_run_at_utc",
        "assumed_available_at_utc",
        "forecast_valid_at_utc",
        "forecast_hour",
    ]]
)

assert forecast_selection.iloc[0][
    "forecast_hour"
] == 32

assert forecast_selection.iloc[1][
    "forecast_hour"
] == 27

assert forecast_selection.iloc[4][
    "forecast_hour"
] == 32

print("\nForecast selection boundary checks passed!")

In [ ]:
def build_weather_patterns(forecast_hour):

    hour = forecast_hour
    instant = f"{hour} hour fcst"

    return {
        "temperature_2m": (
            rf":TMP:2 m above ground:{instant}:"
        ),
        "u_wind_10m": (
            rf":UGRD:10 m above ground:{instant}:"
        ),
        "v_wind_10m": (
            rf":VGRD:10 m above ground:{instant}:"
        ),
        "wind_gust": (
            rf":GUST:surface:{instant}:"
        ),
        "visibility": (
            rf":VIS:surface:{instant}:"
        ),
        "precipitation_1h": (
            rf":APCP:surface:{hour-1}-{hour} hour acc fcst:"
        ),
        "surface_cape": (
            rf":CAPE:surface:{instant}:"
        ),
        "composite_reflectivity": (
            rf":REFC:entire atmosphere:{instant}:"
        ),
        "cloud_ceiling_raw": (
            rf":HGT:cloud ceiling:{instant}:"
        ),
    }

In [ ]:
def parse_grib_inventory(lines):

    records = []

    for line in lines:

        parts = line.split(":", 2)

        if len(parts) != 3:
            raise ValueError(
                f"Unexpected inventory line: {line}"
            )

        records.append({
            "message": parts[0],
            "start_byte": int(parts[1]),
            "inventory": line,
        })

    inventory = pd.DataFrame(records)

    assert not inventory.empty

    assert inventory["start_byte"].is_unique

    assert inventory["start_byte"].is_monotonic_increasing

    inventory["end_byte"] = (
        inventory["start_byte"].shift(-1) - 1
    ).astype("Int64")

    return inventory

In [ ]:
pilot_reference = select_hrrr_forecast(
    "2024-07-15T12:00:00Z"
)

with urlopen(
    pilot_reference["index_url"],
    timeout=30,
) as response:

    pilot_inventory = parse_grib_inventory(
        response.read()
        .decode("utf-8")
        .splitlines()
    )

patterns = build_weather_patterns(
    pilot_reference["forecast_hour"]
)

selected_records = []

for feature, pattern in patterns.items():

    matches = pilot_inventory.loc[
        pilot_inventory["inventory"].str.contains(
            pattern,
            regex=True,
        )
    ]

    assert len(matches) == 1, (
        f"{feature}: expected one match, "
        f"found {len(matches)}"
    )

    record = matches.iloc[0].to_dict()

    record["feature"] = feature

    selected_records.append(record)

selected_messages = pd.DataFrame(
    selected_records
)

assert selected_messages["end_byte"].notna().all()

selected_messages["size_bytes"] = (
    selected_messages["end_byte"]
    - selected_messages["start_byte"]
    + 1
)

display(selected_messages)

print(
    "Selected messages:",
    len(selected_messages),
)

print(
    "Estimated subset size:",
    round(
        selected_messages["size_bytes"].sum()
        / 1024**2,
        2,
    ),
    "MiB",
)

print("\nPrecise message selection passed!")

In [ ]:
def download_grib_message(
    url,
    start_byte,
    end_byte,
):

    start_byte = int(start_byte)
    end_byte = int(end_byte)

    expected_size = end_byte - start_byte + 1

    if expected_size <= 0:
        raise ValueError("Invalid byte range")

    request = Request(
        url,
        headers={
            "Range": (
                f"bytes={start_byte}-{end_byte}"
            ),
            "Accept-Encoding": "identity",
        },
    )

    with urlopen(
        request,
        timeout=60,
    ) as response:

        if response.status != 206:
            raise RuntimeError(
                "Server did not return partial content"
            )

        content_range = response.headers.get(
            "Content-Range", ""
        )

        expected_prefix = (
            f"bytes {start_byte}-{end_byte}/"
        )

        assert content_range.startswith(
            expected_prefix
        )

        payload = response.read()

    assert len(payload) == expected_size

    assert payload[:4] == b"GRIB"

    assert payload[-4:] == b"7777"

    return payload

In [ ]:
def decode_airport_weather(
    payload,
    feature,
    reference,
    airports,
):

    gid = eccodes.codes_new_from_message(
        payload
    )

    try:

        def get(key):
            return eccodes.codes_get(gid, key)

        actual_run = pd.to_datetime(
            f"{get('dataDate')}"
            f"{int(get('dataTime')):04d}",
            format="%Y%m%d%H%M",
            utc=True,
        )

        actual_valid = pd.to_datetime(
            f"{get('validityDate')}"
            f"{int(get('validityTime')):04d}",
            format="%Y%m%d%H%M",
            utc=True,
        )

        assert actual_run == (
            reference["forecast_run_at_utc"]
        )

        assert actual_valid == (
            reference["forecast_valid_at_utc"]
        )

        end_step = int(get("endStep"))
        start_step = int(get("startStep"))

        assert end_step == reference["forecast_hour"]

        if feature == "precipitation_1h":

            assert start_step == end_step - 1

        metadata = {
            "feature": feature,
            "short_name": get("shortName"),
            "name": get("name"),
            "units": get("units"),
            "type_of_level": get("typeOfLevel"),
            "step_type": get("stepType"),
            "start_step": start_step,
            "end_step": end_step,
        }

        bitmap_present = bool(
            get("bitmapPresent")
        )
        
        bitmap = (
            eccodes.codes_get_array(gid, "bitmap")
            if bitmap_present
            else None
        )

        records = []

        for airport in airports.itertuples(
            index=False
        ):

            nearest = eccodes.codes_grib_find_nearest(
                gid,
                airport.Latitude,
                airport.Longitude,
            )[0]

            value = float(nearest["value"])

            if bitmap is not None:
            
                grid_index = int(nearest["index"])
            
                assert 0 <= grid_index < len(bitmap)
            
                if bitmap[grid_index] == 0:
                    value = np.nan
            
                    if not np.isfinite(value):
                        value = np.nan

            records.append({
                "Origin": airport.Origin,
                "feature": feature,
                "value": value,
                "grid_latitude": nearest["lat"],
                "grid_longitude": nearest["lon"],
                "distance_km": nearest["distance"],
                "forecast_run_at_utc": actual_run,
                "forecast_valid_at_utc": actual_valid,
                "forecast_hour": end_step,
            })

        return pd.DataFrame(records), metadata

    finally:
        eccodes.codes_release(gid)

In [ ]:
weather_parts = []
metadata_records = []

downloaded_bytes = 0

start = time.perf_counter()

for message in selected_messages.itertuples(
    index=False
):

    payload = download_grib_message(
        url=pilot_reference["grib_url"],
        start_byte=message.start_byte,
        end_byte=message.end_byte,
    )

    downloaded_bytes += len(payload)

    airport_values, metadata = (
        decode_airport_weather(
            payload=payload,
            feature=message.feature,
            reference=pilot_reference,
            airports=AIRPORT_COORDINATES,
        )
    )

    weather_parts.append(airport_values)

    metadata_records.append(metadata)

    print(
        f"{message.feature} | "
        f"downloaded={len(payload)/1024**2:.2f} MiB | "
        f"airports={len(airport_values)}"
    )

    del payload

weather_long = pd.concat(
    weather_parts,
    ignore_index=True,
)

weather_metadata = pd.DataFrame(
    metadata_records
)

assert len(weather_long) == 6 * 9

assert not weather_long.duplicated(
    ["Origin", "feature"]
).any()

print("\n=== Weather Extraction Summary ===")

print("Extracted observations:", len(weather_long))

print(
    "Downloaded MiB:",
    round(downloaded_bytes / 1024**2, 2),
)

print(
    "Extraction seconds:",
    round(time.perf_counter() - start, 2),
)

display(weather_metadata)

display(weather_long)

print("\nHRRR weather extraction pilot completed!")

In [ ]:
weather_wide = weather_long.pivot(
    index="Origin",
    columns="feature",
    values="value",
).reset_index()

weather_wide.columns.name = None

assert weather_wide.shape == (6, 10)

weather_wide["temperature_c"] = (
    weather_wide["temperature_2m"] - 273.15
)

weather_wide["wind_speed_10m"] = np.hypot(
    weather_wide["u_wind_10m"],
    weather_wide["v_wind_10m"],
)

weather_wide["visibility_km"] = (
    weather_wide["visibility"] / 1000
)

display(weather_wide.round(3))

In [ ]:
print("=== Spatial Quality Audit ===")

display(
    weather_long.groupby("Origin")[
        "distance_km"
    ].agg(["min", "max", "mean"])
)

assert weather_long["distance_km"].ge(0).all()

assert weather_long["distance_km"].le(5).all()

print("\n=== Missing Weather Values ===")

display(
    weather_wide.isna().sum()
)

print("\n=== Weather Metadata ===")

display(
    weather_metadata[[
        "feature",
        "units",
        "type_of_level",
        "step_type",
        "start_step",
        "end_step",
    ]]
)

assert weather_long[
    "forecast_run_at_utc"
].nunique() == 1

assert weather_long[
    "forecast_valid_at_utc"
].nunique() == 1

assert weather_wide[
    "temperature_c"
].dropna().between(-70, 65).all()

for feature in [
    "wind_speed_10m",
    "wind_gust",
    "visibility",
    "precipitation_1h",
    "surface_cape",
]:

    assert weather_wide[
        feature
    ].dropna().ge(0).all()

print("\n=== Airport Weather Forecast ===")

display(weather_wide.round(3))

print("\nWeather pilot quality checks passed!")

## 3. Continuous Weather Forecast Extraction

Extend the single-hour HRRR pilot to a continuous
24-hour period.

### Objectives

- Verify coverage across multiple forecast initialization cycles.
- Validate forecast lead times from 27 to 32 hours.
- Check consistent weather variable definitions and units.
- Preserve structural missing weather values.
- Cache extracted airport-level forecasts.
- Measure network transfer and extraction costs.

Only archived forecasts satisfying the assumed
T-24h availability constraint are permitted.

In [ ]:
RAW_WEATHER_COLUMNS = list(
    build_weather_patterns(30)
)

MODEL_WEATHER_COLUMNS = [
    "origin_temperature_c",
    "origin_wind_speed_10m",
    "origin_wind_gust",
    "origin_visibility_km",
    "origin_precipitation_1h",
    "origin_surface_cape",
    "origin_composite_reflectivity",
]

EXPECTED_METADATA = (
    weather_metadata.set_index("feature")[
        [
            "short_name",
            "units",
            "type_of_level",
            "step_type",
        ]
    ].to_dict("index")
)

print("Raw weather variables:", len(RAW_WEATHER_COLUMNS))

print("Model weather features:", len(MODEL_WEATHER_COLUMNS))

display(pd.DataFrame(EXPECTED_METADATA).T)

In [ ]:
pilot_hours = pd.date_range(
    start="2024-07-15T00:00:00Z",
    periods=24,
    freq="h",
)

daily_references = [
    select_hrrr_forecast(hour)
    for hour in pilot_hours
]

daily_request_plan = pd.DataFrame(
    daily_references
)

assert len(daily_request_plan) == 24

assert daily_request_plan["grib_url"].is_unique

assert daily_request_plan["forecast_hour"].between(
    27, 32
).all()

assert (
    daily_request_plan["assumed_available_at_utc"]
    <= daily_request_plan["prediction_at_utc"]
).all()

display(
    daily_request_plan[
        [
            "forecast_run_at_utc",
            "forecast_valid_at_utc",
            "forecast_hour",
            "assumed_available_at_utc",
            "prediction_at_utc",
        ]
    ]
)

print("\n=== Forecast Lead Distribution ===")

display(
    daily_request_plan["forecast_hour"]
    .value_counts()
    .sort_index()
)

print("\nDaily forecast request plan passed!")

In [ ]:
def extract_forecast_hour(reference):

    with urlopen(
        reference["index_url"],
        timeout=30,
    ) as response:

        inventory = parse_grib_inventory(
            response.read()
            .decode("utf-8")
            .splitlines()
        )

    patterns = build_weather_patterns(
        reference["forecast_hour"]
    )

    parts = []
    downloaded_bytes = 0

    for feature, pattern in patterns.items():

        matches = inventory.loc[
            inventory["inventory"].str.contains(
                pattern,
                regex=True,
            )
        ]

        if len(matches) != 1:
            raise ValueError(
                f"{feature}: expected one GRIB message, "
                f"found {len(matches)}"
            )

        message = matches.iloc[0]

        if pd.isna(message["end_byte"]):
            raise ValueError(
                f"Unresolved byte range: {feature}"
            )

        payload = download_grib_message(
            url=reference["grib_url"],
            start_byte=message["start_byte"],
            end_byte=message["end_byte"],
        )

        downloaded_bytes += len(payload)

        values, metadata = decode_airport_weather(
            payload=payload,
            feature=feature,
            reference=reference,
            airports=AIRPORT_COORDINATES,
        )

        # Require consistent GRIB variable definitions.
        for key, expected in EXPECTED_METADATA[
            feature
        ].items():

            if metadata[key] != expected:
                raise ValueError(
                    f"Metadata mismatch: {feature}, {key}"
                )

        for key in [
            "units",
            "short_name",
            "type_of_level",
            "step_type",
            "start_step",
            "end_step",
        ]:
            values[key] = metadata[key]

        parts.append(values)

        del payload

    result = pd.concat(
        parts,
        ignore_index=True,
    )

    result["assumed_available_at_utc"] = (
        reference["assumed_available_at_utc"]
    )

    result["source_url"] = reference["grib_url"]

    result["availability_policy"] = "research_proxy"

    return result, downloaded_bytes

In [ ]:
import hashlib
import json

cache_config = {
    "version": "hrrr_airport_weather_v1",
    "model": WEATHER_MODEL,
    "product": WEATHER_PRODUCT,
    "availability_lag_hours": (
        WEATHER_AVAILABILITY_LAG_HOURS
    ),
    "availability_policy": "research_proxy",
    "variables": RAW_WEATHER_COLUMNS,
    "expected_metadata": EXPECTED_METADATA,
    "airports": AIRPORT_COORDINATES.to_dict("records"),
}

config_hash = hashlib.sha256(
    json.dumps(
        cache_config,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:16]

WEATHER_CACHE_DIR = (
    ROOT
    / "data/interim/weather_cache"
    / config_hash
)

WEATHER_CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Weather cache configuration:", config_hash)

print("Cache directory:", WEATHER_CACHE_DIR)

In [ ]:
def validate_hourly_weather(frame, reference):

    expected_keys = {
        (airport, feature)
        for airport in ORIGIN_AIRPORTS
        for feature in RAW_WEATHER_COLUMNS
    }

    actual_keys = set(
        frame[["Origin", "feature"]]
        .itertuples(index=False, name=None)
    )

    assert actual_keys == expected_keys

    assert len(frame) == len(expected_keys)

    assert not frame.duplicated(
        ["Origin", "feature"]
    ).any()

    for column in [
        "forecast_run_at_utc",
        "forecast_valid_at_utc",
        "assumed_available_at_utc",
    ]:

        assert frame[column].eq(
            reference[column]
        ).all()

    assert frame["forecast_hour"].eq(
        reference["forecast_hour"]
    ).all()

    assert frame["source_url"].eq(
        reference["grib_url"]
    ).all()

    assert frame["availability_policy"].eq(
        "research_proxy"
    ).all()

    assert (
        frame["assumed_available_at_utc"]
        <= reference["prediction_at_utc"]
    ).all()

    assert frame["distance_km"].between(
        0, 5
    ).all()

    for feature, expected in EXPECTED_METADATA.items():

        rows = frame.loc[
            frame["feature"].eq(feature)
        ]

        for key, value in expected.items():
            assert rows[key].eq(value).all()

    values = frame["value"].dropna()

    assert np.isfinite(values).all()

    precipitation = frame.loc[
        frame["feature"].eq("precipitation_1h")
    ]

    assert (
        precipitation["end_step"]
        - precipitation["start_step"]
    ).eq(1).all()

In [ ]:
def get_cached_weather(reference):

    run = reference[
        "forecast_run_at_utc"
    ].strftime("%Y%m%d_%H")

    hour = reference["forecast_hour"]

    path = (
        WEATHER_CACHE_DIR
        / f"hrrr_{run}_f{hour:02d}.parquet"
    )

    start = time.perf_counter()

    if path.is_file():

        frame = pd.read_parquet(path)

        validate_hourly_weather(
            frame,
            reference,
        )

        return (
            frame,
            {
                "CacheHit": True,
                "DownloadedBytes": 0,
                "Seconds": time.perf_counter() - start,
            },
        )

    frame, downloaded_bytes = (
        extract_forecast_hour(reference)
    )

    validate_hourly_weather(
        frame,
        reference,
    )

    temporary = path.with_suffix(".tmp")

    try:

        frame.to_parquet(
            temporary,
            index=False,
        )

        restored = pd.read_parquet(temporary)

        pd.testing.assert_frame_equal(
            restored,
            frame.reset_index(drop=True),
        )

        temporary.replace(path)

    finally:
        temporary.unlink(missing_ok=True)

    return (
        frame,
        {
            "CacheHit": False,
            "DownloadedBytes": downloaded_bytes,
            "Seconds": time.perf_counter() - start,
        },
    )

In [ ]:
from urllib.error import URLError

daily_weather_parts = []
daily_audit_records = []

start = time.perf_counter()

for reference in daily_references:

    valid_at = reference[
        "forecast_valid_at_utc"
    ]

    try:

        frame, stats = get_cached_weather(
            reference
        )

        daily_weather_parts.append(frame)

        daily_audit_records.append({
            "ValidAtUTC": valid_at,
            "ForecastHour": reference["forecast_hour"],
            "Status": "success",
            **stats,
        })

        print(
            f"{valid_at} | "
            f"lead={reference['forecast_hour']}h | "
            f"cache={stats['CacheHit']} | "
            f"MiB={stats['DownloadedBytes']/1024**2:.2f} | "
            f"seconds={stats['Seconds']:.2f}"
        )

    except (URLError, TimeoutError) as exc:

        daily_audit_records.append({
            "ValidAtUTC": valid_at,
            "ForecastHour": reference["forecast_hour"],
            "Status": "failed",
            "Error": str(exc),
        })

        print(f"{valid_at} | NETWORK ERROR: {exc}")

        continue

daily_weather_audit = pd.DataFrame(
    daily_audit_records
)

print("\n=== Daily Extraction Audit ===")

display(daily_weather_audit)

print(
    "Total execution seconds:",
    round(time.perf_counter() - start, 2),
)

assert daily_weather_audit["Status"].eq(
    "success"
).all(), "Incomplete extraction; rerun this cell"

daily_weather_long = pd.concat(
    daily_weather_parts,
    ignore_index=True,
)

print("\nWeather records:", len(daily_weather_long))

assert len(daily_weather_long) == 24 * 6 * 9

print("\nDaily weather extraction completed!")

## 4. Daily Airport Weather Feature Matrix

Transform decoded GRIB messages into an airport-hour
feature matrix.

Convert temperature and visibility into interpretable
units and derive wind speed from horizontal wind components.

Preserve raw cloud ceiling values and distinguish
weather feature completeness from extraction completeness.

In [ ]:
WEATHER_KEYS = [
    "Origin",
    "forecast_run_at_utc",
    "forecast_valid_at_utc",
    "assumed_available_at_utc",
    "forecast_hour",
]

assert not daily_weather_long.duplicated(
    WEATHER_KEYS + ["feature"]
).any()

daily_weather_wide = (
    daily_weather_long.pivot(
        index=WEATHER_KEYS,
        columns="feature",
        values="value",
    )
    .reset_index()
)

daily_weather_wide.columns.name = None

assert len(daily_weather_wide) == 144

daily_weather_wide["origin_temperature_c"] = (
    daily_weather_wide["temperature_2m"] - 273.15
)

daily_weather_wide["origin_wind_speed_10m"] = np.hypot(
    daily_weather_wide["u_wind_10m"],
    daily_weather_wide["v_wind_10m"],
)

daily_weather_wide["origin_wind_gust"] = (
    daily_weather_wide["wind_gust"]
)

daily_weather_wide["origin_visibility_km"] = (
    daily_weather_wide["visibility"] / 1000
)

daily_weather_wide["origin_precipitation_1h"] = (
    daily_weather_wide["precipitation_1h"]
)

daily_weather_wide["origin_surface_cape"] = (
    daily_weather_wide["surface_cape"]
)

daily_weather_wide["origin_composite_reflectivity"] = (
    daily_weather_wide["composite_reflectivity"]
)

daily_weather_features = daily_weather_wide[
    WEATHER_KEYS + MODEL_WEATHER_COLUMNS
].copy()

print("Airport-hour records:", len(daily_weather_features))

print("\n=== Model Weather Features ===")

display(daily_weather_features.head(10))

print("\n=== Missing Weather Features ===")

display(
    daily_weather_features[
        MODEL_WEATHER_COLUMNS
    ].isna().sum()
)

display(
    daily_weather_features[
        MODEL_WEATHER_COLUMNS
    ].describe().round(3)
)

In [ ]:
assert daily_weather_features["Origin"].nunique() == 6

assert daily_weather_features[
    "forecast_valid_at_utc"
].nunique() == 24

assert daily_weather_features.groupby(
    "Origin"
).size().eq(24).all()

assert not daily_weather_features.duplicated(
    ["Origin", "forecast_valid_at_utc"]
).any()

assert daily_weather_features[
    "origin_temperature_c"
].dropna().between(-70, 65).all()

for column in [
    "origin_wind_speed_10m",
    "origin_wind_gust",
    "origin_visibility_km",
    "origin_precipitation_1h",
    "origin_surface_cape",
]:

    assert daily_weather_features[
        column
    ].dropna().ge(0).all()

core_complete = daily_weather_features[
    MODEL_WEATHER_COLUMNS
].notna().all(axis=1)

print("=== Daily Weather Coverage ===")

print("Expected airport-hour records:", 144)

print("Extracted airport-hour records:",
      len(daily_weather_features))

print("Complete model feature records:",
      int(core_complete.sum()))

print("Model feature coverage:",
      f"{core_complete.mean():.4%}")

print("Cloud ceiling missing records:",
      daily_weather_wide["cloud_ceiling_raw"].isna().sum())

print("\n=== Weather Extraction Cost ===")

display(
    daily_weather_audit[
        [
            "DownloadedBytes",
            "Seconds",
        ]
    ].sum()
)

print(
    "Total downloaded MiB:",
    round(
        daily_weather_audit["DownloadedBytes"].sum()
        / 1024**2,
        2,
    ),
)

print("\nDaily weather quality audit passed!")

In [ ]:
import matplotlib.pyplot as plt

plot_features = {
    "origin_temperature_c": "Temperature (°C)",
    "origin_wind_gust": "Wind Gust (m/s)",
    "origin_surface_cape": "Surface CAPE (J/kg)",
    "origin_visibility_km": "Visibility (km)",
    "origin_precipitation_1h": "Hourly Precipitation (mm)",
    "origin_composite_reflectivity": "Reflectivity (dBZ)",
}

for column, title in plot_features.items():

    plot_data = daily_weather_features.pivot(
        index="forecast_valid_at_utc",
        columns="Origin",
        values=column,
    )

    ax = plot_data.plot(
        figsize=(12, 4),
        marker="o",
        markersize=3,
    )

    ax.set_title(
        f"July 15, 2024 — HRRR {title}"
    )

    ax.set_xlabel("Forecast Valid Time (UTC)")
    ax.set_ylabel(title)
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
cached_parts = []

start = time.perf_counter()

for reference in daily_references:

    cached, stats = get_cached_weather(
        reference
    )

    assert stats["CacheHit"]

    assert stats["DownloadedBytes"] == 0

    cached_parts.append(cached)

cached_daily_weather = pd.concat(
    cached_parts,
    ignore_index=True,
)

pd.testing.assert_frame_equal(
    cached_daily_weather,
    daily_weather_long,
)

print("Cache hits: 24 / 24")

print("Downloaded bytes: 0")

print(
    "Warm-cache seconds:",
    round(time.perf_counter() - start, 3),
)

print("\nWeather cache regression verification passed!")

## 5. HRRR Zarr Acquisition Optimization

The GRIB2 pilot achieved complete weather feature coverage,
but its network transfer and processing costs make
large-scale historical extraction expensive.

This experiment evaluates the HRRR Zarr archive as an
alternative acquisition backend.

### Validation requirements

- Identical forecast initialization time
- Identical forecast valid time and lead
- Identical spatial grid point
- Consistent variable definitions and units
- Explicit handling of archive precision changes
- No silent replacement of missing chunks with zeros

GRIB2 remains the reference implementation.

In [ ]:
import json

import numcodecs
import pyproj
import s3fs
import xarray as xr
import zarr

fs = s3fs.S3FileSystem(anon=True)

ZARR_BUCKET = "hrrrzarr"

ZARR_PILOT_RUN = pd.Timestamp(
    "2024-07-14T06:00:00Z"
)

ZARR_PILOT_LEADS = list(range(27, 33))

print("Zarr version:", zarr.__version__)
print("Xarray version:", xr.__version__)
print("Forecast run:", ZARR_PILOT_RUN)
print("Forecast leads:", ZARR_PILOT_LEADS)

In [ ]:
def build_zarr_group_path(run_at, level, variable):

    run_at = pd.Timestamp(run_at)

    if run_at.tzinfo is None:
        raise ValueError("Timezone-aware run required")

    run_at = run_at.tz_convert("UTC")

    date = run_at.strftime("%Y%m%d")
    cycle = run_at.strftime("%H")

    return (
        f"{ZARR_BUCKET}/sfc/{date}/"
        f"{date}_{cycle}z_fcst.zarr/"
        f"{level}/{variable}"
    )


def build_zarr_array_path(run_at, level, variable):

    group = build_zarr_group_path(
        run_at,
        level,
        variable,
    )

    return f"{group}/{level}/{variable}"


def read_zarr_json(path):

    return json.loads(fs.cat_file(path))


temperature_path = build_zarr_array_path(
    ZARR_PILOT_RUN,
    "2m_above_ground",
    "TMP",
)

temperature_metadata = read_zarr_json(
    temperature_path + "/.zarray"
)

temperature_attributes = read_zarr_json(
    temperature_path + "/.zattrs"
)

print("=== Zarr Array Metadata ===")
print(json.dumps(temperature_metadata, indent=2))

print("\n=== Zarr Variable Attributes ===")
print(json.dumps(temperature_attributes, indent=2))

In [ ]:
temperature_group = build_zarr_group_path(
    ZARR_PILOT_RUN,
    "2m_above_ground",
    "TMP",
)

temperature_ds = xr.open_zarr(
    fs.get_mapper(temperature_group),
    consolidated=False,
    chunks=None,
    decode_times=True,
    decode_timedelta=True,
)

print("=== Temperature Dataset ===")

display(temperature_ds)

print("\nAvailable variables:")
print(list(temperature_ds.variables))

print("\nDataset dimensions:")
print(dict(temperature_ds.sizes))

### 5.1 Zarr Coordinate Inspection

Inspect the temporal and spatial coordinates of the HRRR
temperature forecast dataset.

Verify coordinate dimensions, data types, attributes,
and boundary values before extracting weather data.

This step establishes the temporal and spatial semantics
required for accurate forecast indexing and airport-grid
alignment.

In [ ]:
coordinate_names = [
    "forecast_reference_time",
    "forecast_period",
    "time",
    "projection_x_coordinate",
    "projection_y_coordinate",
]

coordinate_records = []

for name in coordinate_names:

    coord = temperature_ds[name]
    values = np.asarray(coord.values).reshape(-1)

    print(f"\n=== {name} ===")
    print("Dimensions:", coord.dims)
    print("Shape:", coord.shape)
    print("Dtype:", coord.dtype)
    print("Attributes:", dict(coord.attrs))
    print("First values:", values[:5])
    print("Last values:", values[-5:])

    coordinate_records.append({
        "coordinate": name,
        "dimensions": coord.dims,
        "shape": coord.shape,
        "dtype": str(coord.dtype),
        "first_value": str(values[0]),
        "last_value": str(values[-1]),
    })

coordinate_summary = pd.DataFrame(coordinate_records)

print("\n=== Coordinate Summary ===")
display(coordinate_summary)

### 5.2 Forecast Time and Lead Index Validation

Validate the forecast initialization time and ensure
that each forecast valid time equals the initialization
time plus its corresponding forecast period.

Construct an explicit mapping between forecast lead
hours and Zarr array indices using the archived
coordinate values.

This avoids hard-coded temporal indexing assumptions
and ensures consistent forecast selection between
the GRIB and Zarr acquisition backends.

In [ ]:
# Extract temporal coordinates.

reference_times = pd.to_datetime(
    np.asarray(
        temperature_ds["forecast_reference_time"].values
    ).reshape(-1),
    utc=True,
)

forecast_periods = pd.to_timedelta(
    temperature_ds["forecast_period"].values
)

valid_times = pd.to_datetime(
    temperature_ds["time"].values,
    utc=True,
)

reference_time = reference_times[0]

# Validate coordinate completeness.

assert len(reference_times) == 1
assert len(forecast_periods) == 48
assert len(valid_times) == 48

assert not reference_times.isna().any()
assert not forecast_periods.isna().any()
assert not valid_times.isna().any()

# Validate initialization time.

assert reference_time == ZARR_PILOT_RUN, (
    "Forecast initialization mismatch."
)

# Validate all forecast timestamps.

expected_valid_times = (
    reference_time + forecast_periods
)

assert (valid_times == expected_valid_times).all(), (
    "Forecast valid time mismatch."
)

# Convert forecast periods into hours.

lead_hours_float = np.asarray(
    forecast_periods / pd.Timedelta(hours=1)
)

assert np.isfinite(lead_hours_float).all()

assert np.equal(
    lead_hours_float,
    np.floor(lead_hours_float),
).all()

lead_hours = lead_hours_float.astype(int)

# Validate forecast lead sequence.

assert len(np.unique(lead_hours)) == len(lead_hours)

np.testing.assert_array_equal(
    lead_hours,
    np.arange(1, 49),
)

assert valid_times.is_monotonic_increasing

print("=== Forecast Time Validation ===")
print("Initialization:", reference_time)
print("Forecast count:", len(lead_hours))
print("Minimum lead:", lead_hours.min())
print("Maximum lead:", lead_hours.max())
print("First valid time:", valid_times[0])
print("Last valid time:", valid_times[-1])

print("\nInitialization validation: PASS")
print("Forecast time alignment: PASS")
print("Forecast lead sequence: PASS")

In [ ]:
# Build an explicit lead-to-index mapping.

lead_to_index = {
    int(lead): int(index)
    for index, lead in enumerate(lead_hours)
}

forecast_time_mapping = pd.DataFrame({
    "array_index": np.arange(len(lead_hours)),
    "forecast_lead": lead_hours,
    "forecast_reference_time": reference_time,
    "forecast_valid_time": valid_times,
})

# Select the pilot forecast hours.

pilot_time_mapping = (
    forecast_time_mapping.loc[
        forecast_time_mapping["forecast_lead"].isin(
            ZARR_PILOT_LEADS
        )
    ]
    .reset_index(drop=True)
)

assert len(pilot_time_mapping) == len(ZARR_PILOT_LEADS)

assert all(
    lead in lead_to_index
    for lead in ZARR_PILOT_LEADS
)

print("=== Pilot Forecast Lead Mapping ===")

display(pilot_time_mapping)

print("\nLead-to-index mapping:")

for lead in ZARR_PILOT_LEADS:
    print(f"F{lead:02d} -> index {lead_to_index[lead]}")

print("\nPilot lead mapping validation: PASS")

### 5.3 Airport-to-Zarr Grid Mapping

Transform airport latitude and longitude coordinates
into the native HRRR Lambert Conformal Conic projection.

Identify the nearest spatial grid point for each airport
and determine the corresponding Zarr chunk indices.

Validate spatial resolution, grid coverage, and
airport-to-grid distances before reconciling the
selected locations with the GRIB reference data.

In [ ]:
import pyproj

HRRR_CRS = pyproj.CRS.from_proj4(
    "+proj=lcc "
    "+lat_0=38.5 "
    "+lon_0=-97.5 "
    "+lat_1=38.5 "
    "+lat_2=38.5 "
    "+a=6371229 "
    "+b=6371229 "
    "+units=m "
    "+no_defs"
)

forward_transformer = pyproj.Transformer.from_crs(
    HRRR_CRS.geodetic_crs,
    HRRR_CRS,
    always_xy=True,
)

inverse_transformer = pyproj.Transformer.from_crs(
    HRRR_CRS,
    HRRR_CRS.geodetic_crs,
    always_xy=True,
)

hrrr_geod = HRRR_CRS.get_geod()

print("=== HRRR Projection ===")
print(HRRR_CRS)

In [ ]:
x_coords = np.asarray(
    temperature_ds["projection_x_coordinate"].values
)

y_coords = np.asarray(
    temperature_ds["projection_y_coordinate"].values
)

assert x_coords.shape == (1799,)
assert y_coords.shape == (1059,)

assert np.isfinite(x_coords).all()
assert np.isfinite(y_coords).all()

x_spacing = np.diff(x_coords)
y_spacing = np.diff(y_coords)

assert (x_spacing > 0).all()
assert (y_spacing > 0).all()

np.testing.assert_allclose(
    x_spacing,
    3000.0,
    rtol=0,
    atol=1e-6,
)

np.testing.assert_allclose(
    y_spacing,
    3000.0,
    rtol=0,
    atol=1e-6,
)

print("=== HRRR Spatial Grid ===")
print("X grid size:", len(x_coords))
print("Y grid size:", len(y_coords))
print("Total grid points:", len(x_coords) * len(y_coords))
print("X resolution:", x_spacing.mean(), "m")
print("Y resolution:", y_spacing.mean(), "m")

print("\nSpatial grid validation: PASS")

In [ ]:
airport_grid_records = []

MAX_PROJECTED_DISTANCE_M = np.sqrt(2) * 3000 / 2

assert AIRPORT_COORDINATES["Origin"].is_unique

for row in AIRPORT_COORDINATES.itertuples(index=False):

    airport = row.Origin
    latitude = row.Latitude
    longitude = row.Longitude

    # Transform airport coordinates into HRRR projection.

    airport_x, airport_y = forward_transformer.transform(
        longitude,
        latitude,
    )

    assert np.isfinite(airport_x)
    assert np.isfinite(airport_y)

    # Verify airport coverage.

    assert x_coords.min() <= airport_x <= x_coords.max()
    assert y_coords.min() <= airport_y <= y_coords.max()

    # Find the nearest grid point.

    x_index = int(
        np.argmin(np.abs(x_coords - airport_x))
    )

    y_index = int(
        np.argmin(np.abs(y_coords - airport_y))
    )

    grid_x = float(x_coords[x_index])
    grid_y = float(y_coords[y_index])

    # Calculate projected distance.

    projected_distance_m = float(
        np.hypot(
            grid_x - airport_x,
            grid_y - airport_y,
        )
    )

    assert projected_distance_m <= (
        MAX_PROJECTED_DISTANCE_M + 1e-6
    )

    # Convert the selected grid point back to lat/lon.

    grid_lon, grid_lat = inverse_transformer.transform(
        grid_x,
        grid_y,
    )

    # Calculate geographic distance.

    _, _, geographic_distance_m = hrrr_geod.inv(
        longitude,
        latitude,
        grid_lon,
        grid_lat,
    )

    airport_grid_records.append({
        "airport": airport,
        "airport_latitude": latitude,
        "airport_longitude": longitude,
        "airport_x": airport_x,
        "airport_y": airport_y,
        "x_index": x_index,
        "y_index": y_index,
        "grid_x": grid_x,
        "grid_y": grid_y,
        "grid_latitude": grid_lat,
        "grid_longitude": grid_lon,
        "projected_distance_km": projected_distance_m / 1000,
        "geographic_distance_km": geographic_distance_m / 1000,
    })

airport_zarr_grid = pd.DataFrame(airport_grid_records)

assert len(airport_zarr_grid) == 6
assert airport_zarr_grid["airport"].is_unique

assert set(airport_zarr_grid["airport"]) == set(
    ORIGIN_AIRPORTS
)

print("=== Airport-to-Zarr Grid Mapping ===")

display(airport_zarr_grid.round(6))

print(
    "\nMaximum geographic distance:",
    round(
        airport_zarr_grid["geographic_distance_km"].max(),
        4,
    ),
    "km",
)

print("Airport grid mapping validation: PASS")

In [ ]:
temperature_path = build_zarr_array_path(
    ZARR_PILOT_RUN,
    "2m_above_ground",
    "TMP",
)

temperature_metadata = read_zarr_json(
    temperature_path + "/.zarray"
)

temperature_attributes = read_zarr_json(
    temperature_path + "/.zattrs"
)

array_dims = temperature_attributes.get("_ARRAY_DIMENSIONS")

array_shape = tuple(temperature_metadata["shape"])
chunk_shape = tuple(temperature_metadata["chunks"])

print("=== Temperature Array Layout ===")
print("Dimensions:", array_dims)
print("Array shape:", array_shape)
print("Chunk shape:", chunk_shape)
print("Dtype:", temperature_metadata["dtype"])
print("Compressor:", temperature_metadata["compressor"])

assert array_dims is not None, (
    "Array dimension metadata is missing."
)

assert len(array_dims) == len(array_shape) == len(chunk_shape)

expected_dims = {
    "time",
    "projection_y_coordinate",
    "projection_x_coordinate",
}

assert set(array_dims) == expected_dims

for dim, size in zip(array_dims, array_shape):
    assert size == temperature_ds.sizes[dim]

time_axis = array_dims.index("time")
y_axis = array_dims.index("projection_y_coordinate")
x_axis = array_dims.index("projection_x_coordinate")

time_chunk_size = chunk_shape[time_axis]
y_chunk_size = chunk_shape[y_axis]
x_chunk_size = chunk_shape[x_axis]

airport_zarr_grid["chunk_x"] = (
    airport_zarr_grid["x_index"] // x_chunk_size
)

airport_zarr_grid["chunk_y"] = (
    airport_zarr_grid["y_index"] // y_chunk_size
)

airport_zarr_grid["local_x"] = (
    airport_zarr_grid["x_index"] % x_chunk_size
)

airport_zarr_grid["local_y"] = (
    airport_zarr_grid["y_index"] % y_chunk_size
)

airport_zarr_grid["spatial_chunk_id"] = (
    airport_zarr_grid["chunk_y"].astype(str)
    + "."
    + airport_zarr_grid["chunk_x"].astype(str)
)

assert airport_zarr_grid["local_x"].between(
    0, x_chunk_size - 1
).all()

assert airport_zarr_grid["local_y"].between(
    0, y_chunk_size - 1
).all()

print("\n=== Airport Chunk Mapping ===")

display(
    airport_zarr_grid[[
        "airport",
        "x_index",
        "y_index",
        "chunk_x",
        "chunk_y",
        "local_x",
        "local_y",
        "spatial_chunk_id",
        "geographic_distance_km",
    ]]
)

print(
    "\nUnique spatial chunks:",
    airport_zarr_grid["spatial_chunk_id"].nunique(),
)

print("Time chunk size:", time_chunk_size)
print("Spatial chunk validation: PASS")

#### 5.3.1 GRIB–Zarr Spatial Reconciliation

Compare the Zarr-derived airport grid locations with
the existing GRIB reference data.

Validate forecast identity, airport coverage, latitude,
longitude, and geographic separation between the grid
points selected by both acquisition backends.

Reuse cached GRIB records without additional network
requests to ensure a reproducible spatial comparison.

In [ ]:
GRIB_REFERENCE_LEAD = 30

reference_run = ZARR_PILOT_RUN

reference_valid = (
    reference_run
    + pd.Timedelta(hours=GRIB_REFERENCE_LEAD)
)

reference_cache_path = (
    WEATHER_CACHE_DIR
    / (
        f"hrrr_{reference_run.strftime('%Y%m%d_%H')}"
        f"_f{GRIB_REFERENCE_LEAD:02d}.parquet"
    )
)

assert reference_cache_path.is_file(), (
    f"GRIB cache not found: {reference_cache_path}"
)

grib_reference = pd.read_parquet(
    reference_cache_path
)

grib_temperature_reference = (
    grib_reference.loc[
        grib_reference["feature"].eq("temperature_2m")
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(grib_temperature_reference) == 6

assert grib_temperature_reference["Origin"].is_unique

assert set(
    grib_temperature_reference["Origin"]
) == set(ORIGIN_AIRPORTS)

# Validate forecast identity.

assert pd.to_datetime(
    grib_temperature_reference["forecast_run_at_utc"],
    utc=True,
).eq(reference_run).all()

assert pd.to_datetime(
    grib_temperature_reference["forecast_valid_at_utc"],
    utc=True,
).eq(reference_valid).all()

assert grib_temperature_reference[
    "forecast_hour"
].eq(GRIB_REFERENCE_LEAD).all()

print("=== Cached GRIB Temperature Reference ===")
print("Cache:", reference_cache_path.name)
print("Initialization:", reference_run)
print("Valid time:", reference_valid)
print("Forecast lead:", GRIB_REFERENCE_LEAD)

display(
    grib_temperature_reference[[
        "Origin",
        "value",
        "grid_latitude",
        "grid_longitude",
        "distance_km",
    ]]
)

print("\nGRIB reference validation: PASS")

In [ ]:
def normalize_longitude(longitude):
    return (longitude + 180) % 360 - 180


grib_grid = (
    grib_temperature_reference[[
        "Origin",
        "grid_latitude",
        "grid_longitude",
        "distance_km",
    ]]
    .rename(columns={
        "Origin": "airport",
        "grid_latitude": "grib_latitude",
        "grid_longitude": "grib_longitude",
        "distance_km": "grib_distance_km",
    })
    .copy()
)

zarr_grid = (
    airport_zarr_grid[[
        "airport",
        "grid_latitude",
        "grid_longitude",
        "geographic_distance_km",
    ]]
    .rename(columns={
        "grid_latitude": "zarr_latitude",
        "grid_longitude": "zarr_longitude",
        "geographic_distance_km": "zarr_distance_km",
    })
    .copy()
)

spatial_comparison = zarr_grid.merge(
    grib_grid,
    on="airport",
    how="inner",
    validate="one_to_one",
)

assert len(spatial_comparison) == 6

for column in ["grib_longitude", "zarr_longitude"]:

    spatial_comparison[column] = normalize_longitude(
        spatial_comparison[column]
    )

# Coordinate differences.

spatial_comparison["latitude_error_deg"] = (
    spatial_comparison["zarr_latitude"]
    - spatial_comparison["grib_latitude"]
).abs()

spatial_comparison["longitude_error_deg"] = np.abs(
    normalize_longitude(
        spatial_comparison["zarr_longitude"]
        - spatial_comparison["grib_longitude"]
    )
)

# Geographic distance between the two grid points.

_, _, grid_separation_m = hrrr_geod.inv(
    spatial_comparison["grib_longitude"].to_numpy(),
    spatial_comparison["grib_latitude"].to_numpy(),
    spatial_comparison["zarr_longitude"].to_numpy(),
    spatial_comparison["zarr_latitude"].to_numpy(),
)

spatial_comparison["grid_separation_m"] = (
    grid_separation_m
)

spatial_comparison["airport_distance_error_m"] = (
    (
        spatial_comparison["zarr_distance_km"]
        - spatial_comparison["grib_distance_km"]
    ).abs() * 1000
)

# Allow a small tolerance for coordinate representation.

GRID_MATCH_TOLERANCE_M = 2.0

assert np.isfinite(
    spatial_comparison["grid_separation_m"]
).all()

spatial_comparison["grid_match"] = (
    spatial_comparison["grid_separation_m"]
    <= GRID_MATCH_TOLERANCE_M
)

print("=== GRIB vs. Zarr Spatial Comparison ===")

display(spatial_comparison.round(8))

print(
    "\nMaximum grid separation:",
    spatial_comparison["grid_separation_m"].max(),
    "m",
)

print(
    "Matched airports:",
    spatial_comparison["grid_match"].sum(),
    "/ 6",
)

assert spatial_comparison["grid_match"].all(), (
    "GRIB and Zarr spatial grid mismatch."
)

print("\nSpatial reconciliation: PASS")

### 5.4 Temperature Chunk Extraction

Download only the Zarr chunks required for the six
origin airports.

Decode the compressed temperature arrays using the
archived Zarr metadata and extract airport-level
temperature values for forecast leads F27–F32.

Record the compressed payload size, download time,
and decoding time for the acquisition benchmark.

Missing chunks and invalid values must be explicitly
detected rather than silently replaced with fill values.

In [ ]:
# Inspect the Zarr storage configuration.

zarr_dtype = np.dtype(
    temperature_metadata["dtype"]
)

zarr_chunk_shape = tuple(
    temperature_metadata["chunks"]
)

dimension_separator = temperature_metadata.get(
    "dimension_separator", "."
)

array_order = temperature_metadata["order"]

assert temperature_metadata["zarr_format"] == 2

assert array_dims == [
    "time",
    "projection_y_coordinate",
    "projection_x_coordinate",
]

assert zarr_chunk_shape == (48, 150, 150)
assert dimension_separator in [".", "/"]
assert array_order in ["C", "F"]

print("=== Zarr Storage Configuration ===")
print("Dtype:", zarr_dtype)
print("Chunk shape:", zarr_chunk_shape)
print("Dimension separator:", dimension_separator)
print("Array order:", array_order)
print("Fill value:", temperature_metadata["fill_value"])
print("Filters:", temperature_metadata["filters"])

print("\nTemperature attributes:")
print(json.dumps(temperature_attributes, indent=2))

# Build the chunk request plan.

temperature_chunk_plan = airport_zarr_grid[[
    "airport",
    "chunk_y",
    "chunk_x",
    "local_y",
    "local_x",
]].copy()

# All selected forecast hours belong to time chunk 0.

assert all(
    lead_to_index[lead] // time_chunk_size == 0
    for lead in ZARR_PILOT_LEADS
)

temperature_chunk_plan["chunk_key"] = [
    dimension_separator.join(
        map(str, (0, row.chunk_y, row.chunk_x))
    )
    for row in temperature_chunk_plan.itertuples(
        index=False
    )
]

temperature_chunk_plan["object_path"] = (
    temperature_path
    + "/"
    + temperature_chunk_plan["chunk_key"]
)

print("\n=== Temperature Chunk Request Plan ===")

display(temperature_chunk_plan)

print(
    "\nUnique chunk objects:",
    temperature_chunk_plan["object_path"].nunique()
)

assert len(temperature_chunk_plan) == 6

assert (
    temperature_chunk_plan["object_path"].nunique()
    == 6
)

print("Chunk request plan validation: PASS")

In [ ]:
from time import perf_counter

# Initialize the codec from the actual metadata.

temperature_codec = numcodecs.get_codec(
    temperature_metadata["compressor"]
)

# This pilot supports arrays without additional filters.

assert temperature_metadata["filters"] in (
    None, []
), "Additional Zarr filters require explicit decoding."

expected_chunk_bytes = (
    int(np.prod(zarr_chunk_shape))
    * zarr_dtype.itemsize
)

temperature_chunks = {}
temperature_download_records = []

benchmark_start = perf_counter()

for row in temperature_chunk_plan.itertuples(
    index=False
):

    chunk_key = row.chunk_key
    object_path = row.object_path

    print(
        f"Downloading {row.airport}: {chunk_key}"
    )

    # Download the compressed chunk.

    download_start = perf_counter()

    try:
        compressed_bytes = fs.cat_file(object_path)

    except FileNotFoundError as exc:
        raise RuntimeError(
            f"Missing temperature chunk: {object_path}"
        ) from exc

    download_seconds = (
        perf_counter() - download_start
    )

    assert len(compressed_bytes) > 0

    # Decode the compressed chunk.

    decode_start = perf_counter()

    decoded_bytes = temperature_codec.decode(
        compressed_bytes
    )

    chunk_array = np.frombuffer(
        decoded_bytes,
        dtype=zarr_dtype,
    )

    assert chunk_array.nbytes == expected_chunk_bytes, (
        f"Unexpected decoded size: {chunk_key}"
    )

    chunk_array = chunk_array.reshape(
        zarr_chunk_shape,
        order=array_order,
    )

    decode_seconds = (
        perf_counter() - decode_start
    )

    temperature_chunks[chunk_key] = chunk_array

    temperature_download_records.append({
        "airport": row.airport,
        "chunk_key": chunk_key,
        "compressed_bytes": len(compressed_bytes),
        "decoded_bytes": chunk_array.nbytes,
        "download_seconds": download_seconds,
        "decode_seconds": decode_seconds,
    })

    print(
        f"  Payload: "
        f"{len(compressed_bytes) / 1024**2:.4f} MiB"
    )

    print(
        f"  Download: {download_seconds:.3f}s"
    )

    print(
        f"  Decode: {decode_seconds:.3f}s"
    )

temperature_benchmark_seconds = (
    perf_counter() - benchmark_start
)

temperature_download_audit = pd.DataFrame(
    temperature_download_records
)

assert len(temperature_chunks) == 6

print("\n=== Temperature Chunk Download Audit ===")

display(temperature_download_audit.round(4))

print(
    "\nTotal compressed payload:",
    round(
        temperature_download_audit[
            "compressed_bytes"
        ].sum() / 1024**2,
        4,
    ),
    "MiB"
)

print(
    "Total decoded size:",
    round(
        temperature_download_audit[
            "decoded_bytes"
        ].sum() / 1024**2,
        4,
    ),
    "MiB"
)

print(
    "Total processing time:",
    round(temperature_benchmark_seconds, 3),
    "seconds"
)

print("\nTemperature chunk extraction: PASS")

In [ ]:
temperature_value_records = []

for row in temperature_chunk_plan.itertuples(
    index=False
):

    chunk_array = temperature_chunks[
        row.chunk_key
    ]

    for lead in ZARR_PILOT_LEADS:

        array_index = lead_to_index[lead]

        local_time = (
            array_index % time_chunk_size
        )

        temperature_value = float(
            chunk_array[
                local_time,
                row.local_y,
                row.local_x,
            ]
        )

        temperature_value_records.append({
            "Origin": row.airport,
            "feature": "temperature_2m",
            "value": temperature_value,
            "forecast_run_at_utc": ZARR_PILOT_RUN,
            "forecast_valid_at_utc": (
                valid_times[array_index]
            ),
            "forecast_hour": lead,
            "chunk_key": row.chunk_key,
            "array_index": array_index,
        })

zarr_temperature_long = pd.DataFrame(
    temperature_value_records
)

# Validate completeness and uniqueness.

assert len(zarr_temperature_long) == 6 * 6

assert not zarr_temperature_long.duplicated(
    subset=["Origin", "forecast_hour"]
).any()

assert np.isfinite(
    zarr_temperature_long["value"]
).all()

# Reject explicit finite fill values.

fill_value = temperature_metadata["fill_value"]

if fill_value is not None:

    numeric_fill = float(fill_value)

    if np.isfinite(numeric_fill):

        assert not zarr_temperature_long[
            "value"
        ].eq(numeric_fill).any(), (
            "Explicit fill value detected."
        )

assert set(
    zarr_temperature_long["Origin"]
) == set(ORIGIN_AIRPORTS)

assert set(
    zarr_temperature_long["forecast_hour"]
) == set(ZARR_PILOT_LEADS)

assert zarr_temperature_long.groupby(
    "Origin"
).size().eq(6).all()

assert zarr_temperature_long.groupby(
    "forecast_hour"
).size().eq(6).all()

print("=== Extracted Zarr Temperature Records ===")

display(zarr_temperature_long)

print("\nTotal records:", len(zarr_temperature_long))

print(
    "Temperature units:",
    temperature_attributes.get("units", "Unknown")
)

print(
    "Minimum raw temperature:",
    zarr_temperature_long["value"].min()
)

print(
    "Maximum raw temperature:",
    zarr_temperature_long["value"].max()
)

print("\nAirport temperature extraction: PASS")

In [ ]:
zarr_temperature_pivot = (
    zarr_temperature_long.pivot(
        index="Origin",
        columns="forecast_hour",
        values="value",
    )
    .reindex(ORIGIN_AIRPORTS)
)

zarr_temperature_pivot.columns = [
    f"F{lead:02d}"
    for lead in zarr_temperature_pivot.columns
]

print("=== Zarr Temperature Summary ===")

display(zarr_temperature_pivot.round(6))

print("\nSummary shape:", zarr_temperature_pivot.shape)

assert zarr_temperature_pivot.shape == (6, 6)

assert not zarr_temperature_pivot.isna().any().any()

print("Temperature summary validation: PASS")

### 5.5 GRIB vs. Zarr Numerical Comparison

Compare airport-level temperature forecasts extracted
from Zarr against the existing cached GRIB reference
records for forecast leads F27–F32.

Validate forecast identity, spatial consistency,
record completeness, and numerical equivalence.

Account for the floating-point precision of the
archived Zarr temperature array when evaluating
numerical differences.

In [ ]:
grib_temperature_parts = []
grib_reference_cache_records = []

for lead in ZARR_PILOT_LEADS:

    run_at = ZARR_PILOT_RUN

    valid_at = (
        run_at + pd.Timedelta(hours=lead)
    )

    cache_path = (
        WEATHER_CACHE_DIR
        / (
            f"hrrr_{run_at.strftime('%Y%m%d_%H')}"
            f"_f{lead:02d}.parquet"
        )
    )

    assert cache_path.is_file(), (
        f"Missing GRIB cache: {cache_path}"
    )

    frame = pd.read_parquet(cache_path)

    temperature_frame = (
        frame.loc[
            frame["feature"].eq("temperature_2m")
        ]
        .copy()
        .reset_index(drop=True)
    )

    # Validate completeness.

    assert len(temperature_frame) == 6

    assert temperature_frame["Origin"].is_unique

    assert set(
        temperature_frame["Origin"]
    ) == set(ORIGIN_AIRPORTS)

    # Validate forecast identity.

    assert pd.to_datetime(
        temperature_frame["forecast_run_at_utc"],
        utc=True,
    ).eq(run_at).all()

    assert pd.to_datetime(
        temperature_frame["forecast_valid_at_utc"],
        utc=True,
    ).eq(valid_at).all()

    assert temperature_frame[
        "forecast_hour"
    ].eq(lead).all()

    assert np.isfinite(
        temperature_frame["value"]
    ).all()

    grib_temperature_parts.append(
        temperature_frame
    )

    grib_reference_cache_records.append({
        "forecast_hour": lead,
        "valid_time": valid_at,
        "cache_file": cache_path.name,
        "temperature_records": len(temperature_frame),
        "status": "success",
    })

grib_temperature_long = pd.concat(
    grib_temperature_parts,
    ignore_index=True,
)

grib_reference_cache_audit = pd.DataFrame(
    grib_reference_cache_records
)

assert len(grib_temperature_long) == 36

assert not grib_temperature_long.duplicated(
    subset=["Origin", "forecast_hour"]
).any()

print("=== GRIB Reference Cache Audit ===")

display(grib_reference_cache_audit)

print("\nTotal reference records:", len(grib_temperature_long))
print("GRIB reference completeness: PASS")

In [ ]:
FORECAST_MATCH_KEYS = [
    "Origin",
    "feature",
    "forecast_run_at_utc",
    "forecast_valid_at_utc",
    "forecast_hour",
]

grib_compare = (
    grib_temperature_long[
        FORECAST_MATCH_KEYS
        + [
            "value",
            "grid_latitude",
            "grid_longitude",
        ]
    ]
    .rename(columns={
        "value": "grib_value",
        "grid_latitude": "grib_latitude",
        "grid_longitude": "grib_longitude",
    })
    .copy()
)

zarr_compare = (
    zarr_temperature_long[
        FORECAST_MATCH_KEYS
        + [
            "value",
            "chunk_key",
            "array_index",
        ]
    ]
    .rename(columns={
        "value": "zarr_value",
    })
    .copy()
)

temperature_comparison = grib_compare.merge(
    zarr_compare,
    on=FORECAST_MATCH_KEYS,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

assert len(temperature_comparison) == 36

assert temperature_comparison[
    "_merge"
].eq("both").all(), (
    "Unmatched forecast records detected."
)

temperature_comparison = (
    temperature_comparison
    .drop(columns="_merge")
    .sort_values(["Origin", "forecast_hour"])
    .reset_index(drop=True)
)

print("=== Forecast Identity Comparison ===")

display(
    temperature_comparison[[
        "Origin",
        "forecast_hour",
        "forecast_valid_at_utc",
        "grib_value",
        "zarr_value",
    ]]
)

print("\nMatched records:", len(temperature_comparison))
print("Forecast identity reconciliation: PASS")

In [ ]:
zarr_grid_reference = (
    airport_zarr_grid[[
        "airport",
        "grid_latitude",
        "grid_longitude",
    ]]
    .rename(columns={
        "airport": "Origin",
        "grid_latitude": "zarr_latitude",
        "grid_longitude": "zarr_longitude",
    })
)

temperature_comparison = temperature_comparison.merge(
    zarr_grid_reference,
    on="Origin",
    how="left",
    validate="many_to_one",
)

_, _, separation_m = hrrr_geod.inv(
    normalize_longitude(
        temperature_comparison["grib_longitude"]
    ).to_numpy(),
    temperature_comparison["grib_latitude"].to_numpy(),
    temperature_comparison["zarr_longitude"].to_numpy(),
    temperature_comparison["zarr_latitude"].to_numpy(),
)

temperature_comparison["grid_separation_m"] = (
    separation_m
)

assert np.isfinite(separation_m).all()

assert (
    separation_m <= GRID_MATCH_TOLERANCE_M
).all()

print("=== Full Spatial Reconciliation ===")

print("Matched records:", len(temperature_comparison))

print(
    "Maximum grid separation:",
    temperature_comparison["grid_separation_m"].max(),
    "m",
)

print("Full spatial reconciliation: PASS")

In [ ]:
assert temperature_attributes.get("units") == "K"

storage_dtype = np.dtype(
    temperature_metadata["dtype"]
)

assert np.issubdtype(
    storage_dtype,
    np.floating,
)

grib_values = temperature_comparison[
    "grib_value"
].to_numpy(dtype=np.float64)

zarr_values = temperature_comparison[
    "zarr_value"
].to_numpy(dtype=np.float64)

# Simulate the expected Zarr storage precision.

expected_zarr_values = (
    grib_values
    .astype(storage_dtype)
    .astype(np.float64)
)

assert np.isfinite(expected_zarr_values).all()

temperature_comparison["expected_zarr_value"] = (
    expected_zarr_values
)

# Raw numerical differences.

temperature_comparison["signed_error_k"] = (
    zarr_values - grib_values
)

temperature_comparison["absolute_error_k"] = np.abs(
    zarr_values - grib_values
)

# Error caused by floating-point conversion alone.

temperature_comparison["quantization_error_k"] = np.abs(
    expected_zarr_values - grib_values
)

# Remaining error after adjusting for storage precision.

temperature_comparison["precision_adjusted_error_k"] = (
    np.abs(zarr_values - expected_zarr_values)
)

temperature_comparison["raw_exact_match"] = (
    zarr_values == grib_values
)

temperature_comparison["precision_adjusted_match"] = (
    zarr_values == expected_zarr_values
)

print("=== Temperature Numerical Comparison ===")

display(
    temperature_comparison[[
        "Origin",
        "forecast_hour",
        "grib_value",
        "zarr_value",
        "absolute_error_k",
        "quantization_error_k",
        "precision_adjusted_error_k",
        "precision_adjusted_match",
    ]]
)

print("\nStorage dtype:", storage_dtype)

print(
    "Raw exact matches:",
    temperature_comparison["raw_exact_match"].sum(),
    "/ 36",
)

print(
    "Precision-adjusted matches:",
    temperature_comparison[
        "precision_adjusted_match"
    ].sum(),
    "/ 36",
)

In [ ]:
absolute_errors = temperature_comparison[
    "absolute_error_k"
].to_numpy()

precision_adjusted_errors = temperature_comparison[
    "precision_adjusted_error_k"
].to_numpy()

temperature_validation_summary = pd.DataFrame([{
    "variable": "temperature_2m",
    "storage_dtype": str(storage_dtype),
    "compared_records": len(temperature_comparison),

    "raw_exact_matches": int(
        temperature_comparison[
            "raw_exact_match"
        ].sum()
    ),

    "precision_adjusted_matches": int(
        temperature_comparison[
            "precision_adjusted_match"
        ].sum()
    ),

    "mean_absolute_error_k": float(
        absolute_errors.mean()
    ),

    "rmse_k": float(
        np.sqrt(np.mean(absolute_errors ** 2))
    ),

    "max_absolute_error_k": float(
        absolute_errors.max()
    ),

    "max_precision_adjusted_error_k": float(
        precision_adjusted_errors.max()
    ),

    "max_grid_separation_m": float(
        temperature_comparison[
            "grid_separation_m"
        ].max()
    ),
}])

print("=== Temperature Validation Summary ===")

display(temperature_validation_summary.T)

print("\n=== Error by Forecast Lead ===")

display(
    temperature_comparison.groupby(
        "forecast_hour"
    ).agg(
        records=("Origin", "size"),
        mae_k=("absolute_error_k", "mean"),
        max_error_k=("absolute_error_k", "max"),
        adjusted_max_error_k=(
            "precision_adjusted_error_k",
            "max",
        ),
    )
)

assert temperature_comparison[
    "precision_adjusted_match"
].all(), (
    "Numerical differences remain after "
    "storage precision adjustment."
)

print("\nGRIB–Zarr numerical reconciliation: PASS")

### 5.6 Temperature Acquisition Benchmark

Benchmark the GRIB and Zarr acquisition backends
under the same temperature extraction workload.

Both backends must produce 36 airport-level temperature
forecasts covering six airports and forecast leads
F27–F32 from the same initialization cycle.

Measure downloaded payload size, data acquisition time,
decoding time, and processing time.

Separate metadata acquisition from weather payload
processing to ensure a transparent performance comparison.

In [ ]:
from time import perf_counter

grib_benchmark_references = sorted(
    [
        reference
        for reference in daily_references
        if (
            reference["forecast_run_at_utc"]
            == ZARR_PILOT_RUN
            and reference["forecast_hour"]
            in ZARR_PILOT_LEADS
        )
    ],
    key=lambda reference: reference["forecast_hour"],
)

assert len(grib_benchmark_references) == 6

assert [
    reference["forecast_hour"]
    for reference in grib_benchmark_references
] == ZARR_PILOT_LEADS

grib_benchmark_plan = pd.DataFrame(
    grib_benchmark_references
)

print("=== GRIB Temperature Benchmark Plan ===")

display(
    grib_benchmark_plan[[
        "forecast_run_at_utc",
        "forecast_valid_at_utc",
        "forecast_hour",
        "grib_url",
    ]]
)

print("Forecast files:", len(grib_benchmark_plan))

print("Weather variables: 1")
print("Expected temperature records: 36")

print("\nBenchmark request plan: PASS")

In [ ]:
grib_benchmark_records = []
grib_benchmark_parts = []

grib_benchmark_start = perf_counter()

for reference in grib_benchmark_references:

    lead = reference["forecast_hour"]
    feature = "temperature_2m"

    print(f"Benchmarking GRIB F{lead:02d}")

    # 1. Retrieve and parse inventory metadata.

    metadata_start = perf_counter()

    with urlopen(
        reference["index_url"],
        timeout=30,
    ) as response:

        index_payload = response.read()

    inventory = parse_grib_inventory(
        index_payload.decode("utf-8").splitlines()
    )

    pattern = build_weather_patterns(lead)[feature]

    matches = inventory.loc[
        inventory["inventory"].str.contains(
            pattern,
            regex=True,
        )
    ]

    assert len(matches) == 1

    message = matches.iloc[0]

    assert pd.notna(message["end_byte"])

    metadata_seconds = (
        perf_counter() - metadata_start
    )

    # 2. Download the temperature GRIB message.

    download_start = perf_counter()

    payload = download_grib_message(
        url=reference["grib_url"],
        start_byte=message["start_byte"],
        end_byte=message["end_byte"],
    )

    download_seconds = (
        perf_counter() - download_start
    )

    # 3. Decode airport-level temperature values.

    decode_start = perf_counter()

    values, metadata = decode_airport_weather(
        payload=payload,
        feature=feature,
        reference=reference,
        airports=AIRPORT_COORDINATES,
    )

    decode_seconds = (
        perf_counter() - decode_start
    )

    # 4. Validate decoded records.

    for key, expected in EXPECTED_METADATA[
        feature
    ].items():

        assert metadata[key] == expected

    assert metadata["units"] == "K"
    assert len(values) == 6
    assert values["Origin"].is_unique

    assert set(values["Origin"]) == set(
        ORIGIN_AIRPORTS
    )

    assert np.isfinite(values["value"]).all()

    grib_benchmark_parts.append(values)

    grib_benchmark_records.append({
        "forecast_hour": lead,
        "metadata_bytes": len(index_payload),
        "payload_bytes": len(payload),
        "metadata_seconds": metadata_seconds,
        "download_seconds": download_seconds,
        "decode_seconds": decode_seconds,
        "temperature_records": len(values),
    })

    print(
        f"  Payload: "
        f"{len(payload) / 1024**2:.4f} MiB"
    )

    print(f"  Metadata: {metadata_seconds:.3f}s")
    print(f"  Download: {download_seconds:.3f}s")
    print(f"  Decode: {decode_seconds:.3f}s")

grib_benchmark_seconds = (
    perf_counter() - grib_benchmark_start
)

grib_benchmark_audit = pd.DataFrame(
    grib_benchmark_records
)

grib_benchmark_long = pd.concat(
    grib_benchmark_parts,
    ignore_index=True,
)

assert len(grib_benchmark_long) == 36

print("\n=== GRIB Temperature Benchmark Audit ===")

display(grib_benchmark_audit.round(4))

print(
    "\nTotal payload:",
    grib_benchmark_audit["payload_bytes"].sum() / 1024**2,
    "MiB",
)

print(
    "Total metadata time:",
    grib_benchmark_audit["metadata_seconds"].sum(),
    "seconds",
)

print(
    "Total processing time:",
    grib_benchmark_seconds,
    "seconds",
)

print("\nGRIB temperature benchmark: PASS")

In [ ]:
benchmark_reconciliation = (
    grib_benchmark_long[
        FORECAST_MATCH_KEYS + ["value"]
    ]
    .merge(
        zarr_temperature_long[
            FORECAST_MATCH_KEYS + ["value"]
        ],
        on=FORECAST_MATCH_KEYS,
        how="outer",
        suffixes=("_grib", "_zarr"),
        validate="one_to_one",
        indicator=True,
    )
)

assert len(benchmark_reconciliation) == 36

assert benchmark_reconciliation[
    "_merge"
].eq("both").all()

benchmark_reconciliation["absolute_error_k"] = (
    benchmark_reconciliation["value_grib"]
    - benchmark_reconciliation["value_zarr"]
).abs()

assert benchmark_reconciliation[
    "absolute_error_k"
].eq(0).all()

print("=== Benchmark Output Validation ===")

print(
    "Matched records:",
    len(benchmark_reconciliation),
)

print(
    "Maximum absolute error:",
    benchmark_reconciliation[
        "absolute_error_k"
    ].max(),
    "K",
)

print("Benchmark output equivalence: PASS")

In [ ]:
temperature_benchmark_summary = pd.DataFrame([
    {
        "backend": "GRIB",
        "temperature_records": len(
            grib_benchmark_long
        ),
        "weather_payload_requests": len(
            grib_benchmark_audit
        ),
        "payload_mib": (
            grib_benchmark_audit[
                "payload_bytes"
            ].sum() / 1024**2
        ),
        "download_seconds": (
            grib_benchmark_audit[
                "download_seconds"
            ].sum()
        ),
        "decode_seconds": (
            grib_benchmark_audit[
                "decode_seconds"
            ].sum()
        ),
        "metadata_seconds": (
            grib_benchmark_audit[
                "metadata_seconds"
            ].sum()
        ),
        "reported_processing_seconds": (
            grib_benchmark_seconds
        ),
    },
    {
        "backend": "Zarr",
        "temperature_records": len(
            zarr_temperature_long
        ),
        "weather_payload_requests": len(
            temperature_download_audit
        ),
        "payload_mib": (
            temperature_download_audit[
                "compressed_bytes"
            ].sum() / 1024**2
        ),
        "download_seconds": (
            temperature_download_audit[
                "download_seconds"
            ].sum()
        ),
        "decode_seconds": (
            temperature_download_audit[
                "decode_seconds"
            ].sum()
        ),
        "metadata_seconds": np.nan,
        "reported_processing_seconds": (
            temperature_benchmark_seconds
        ),
    },
]).set_index("backend")

temperature_benchmark_summary[
    "payload_processing_seconds"
] = (
    temperature_benchmark_summary["download_seconds"]
    + temperature_benchmark_summary["decode_seconds"]
)

grib_stats = temperature_benchmark_summary.loc["GRIB"]
zarr_stats = temperature_benchmark_summary.loc["Zarr"]

payload_reduction_pct = (
    1 - zarr_stats["payload_mib"]
    / grib_stats["payload_mib"]
) * 100

payload_processing_speedup = (
    grib_stats["payload_processing_seconds"]
    / zarr_stats["payload_processing_seconds"]
)

print("=== Temperature Acquisition Benchmark ===")

display(
    temperature_benchmark_summary.T.round(4)
)

print(
    "\nPayload reduction:",
    f"{payload_reduction_pct:.2f}%",
)

print(
    "Payload processing speedup:",
    f"{payload_processing_speedup:.2f}x",
)

print(
    "\nNote: Metadata acquisition is excluded "
    "from the speedup calculation."
)

print(
    "Zarr metadata acquisition time was not "
    "measured in the previous pilot."
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))

temperature_benchmark_summary[
    "payload_mib"
].plot.bar(
    ax=ax,
    rot=0,
)

ax.set_title("Temperature Acquisition Payload")
ax.set_ylabel("Downloaded Weather Payload (MiB)")
ax.set_xlabel("Acquisition Backend")

ax.bar_label(
    ax.containers[0],
    fmt="%.2f",
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

temperature_benchmark_summary[[
    "download_seconds",
    "decode_seconds",
]].plot.bar(
    stacked=True,
    ax=ax,
    rot=0,
)

ax.set_title("Temperature Payload Processing Time")
ax.set_ylabel("Time (seconds)")
ax.set_xlabel("Acquisition Backend")

ax.legend([
    "Download",
    "Decode / GRIB Point Extraction",
])

plt.tight_layout()
plt.show()

#### 5.6.1 Temperature Benchmark Findings

The Zarr backend reproduced all 36 GRIB reference
temperature values with exact numerical equality.

For the selected forecast cycle and six-airport workload,
Zarr achieved a 17.49x payload processing speedup,
reducing measured processing time from 18.68 seconds
to 1.07 seconds.

However, Zarr transferred 13.49% more weather payload
than GRIB because each spatial chunk contained all
48 forecast lead hours.

The observed performance advantage was primarily driven
by reduced decoding and spatial extraction overhead,
rather than reduced data transfer volume.

These results represent a single temperature pilot,
not an end-to-end or multi-variable performance guarantee.

### 5.7 Multi-Variable Zarr Feasibility

Extend the validated temperature acquisition approach
to the remaining HRRR weather variables.

Evaluate variable availability, storage layout, temporal
coordinates, units, numerical precision, and missing
data semantics.

Retain GRIB as the reference implementation and fallback
backend for unsupported or incompatible variables.

#### 5.7.1 Variable Discovery

Discover the actual HRRR Zarr storage paths for all
required raw weather variables.

Inspect array metadata before attempting multi-variable
chunk extraction.

Variable availability alone does not establish numerical
equivalence or compatibility with the existing weather
feature contract.

In [ ]:
ZARR_VARIABLE_REQUIREMENTS = pd.DataFrame([
    ("temperature_2m", "2m_above_ground", "TMP"),
    ("u_wind_10m", "10m_above_ground", "UGRD"),
    ("v_wind_10m", "10m_above_ground", "VGRD"),
    ("wind_gust", "surface", "GUST"),
    ("visibility", "surface", "VIS"),
    ("precipitation_1h", "surface", "APCP_1hr_acc_fcst"),
    ("surface_cape", "surface", "CAPE"),
    ("composite_reflectivity", "entire_atmosphere", "REFC"),
    ("cloud_ceiling_raw", "cloud_ceiling", "HGT"),
], columns=[
    "feature",
    "candidate_level",
    "variable",
])

assert len(ZARR_VARIABLE_REQUIREMENTS) == 9

assert ZARR_VARIABLE_REQUIREMENTS["feature"].is_unique

print("=== Zarr Variable Requirements ===")

display(ZARR_VARIABLE_REQUIREMENTS)

In [ ]:
precipitation_requirement = (
    ZARR_VARIABLE_REQUIREMENTS.loc[
        ZARR_VARIABLE_REQUIREMENTS["feature"]
        .eq("precipitation_1h")
    ]
)

display(precipitation_requirement)

assert len(precipitation_requirement) == 1

assert (
    precipitation_requirement["variable"].iloc[0]
    == "APCP_1hr_acc_fcst"
)

print("Precipitation variable path correction: PASS")

In [ ]:
precipitation_path = build_zarr_array_path(
    ZARR_PILOT_RUN,
    "surface",
    "APCP_1hr_acc_fcst",
)

print("=== Hourly Precipitation Array Path ===")
print(precipitation_path)

precipitation_metadata = read_zarr_json(
    precipitation_path + "/.zarray"
)

precipitation_attributes = read_zarr_json(
    precipitation_path + "/.zattrs"
)

print("\n=== Precipitation Metadata ===")

print(
    json.dumps(
        precipitation_metadata,
        indent=2,
    )
)

print("\n=== Precipitation Attributes ===")

print(
    json.dumps(
        precipitation_attributes,
        indent=2,
    )
)

print("\nPrecipitation metadata retrieval: PASS")

In [ ]:
# Derive the run-level path from the verified group path.

zarr_run_path = temperature_group.rsplit("/", 2)[0]

print("=== HRRR Zarr Forecast Archive ===")
print("Run path:", zarr_run_path)

run_objects = fs.ls(
    zarr_run_path,
    detail=True,
)

available_levels = sorted([
    item["name"].rstrip("/").rsplit("/", 1)[-1]
    for item in run_objects
    if item["type"] == "directory"
])

assert "2m_above_ground" in available_levels

print("\n=== Available Forecast Levels ===")

for level in available_levels:
    print(level)

print("\nTotal levels:", len(available_levels))

In [ ]:
required_levels = set(
    ZARR_VARIABLE_REQUIREMENTS["candidate_level"]
)

relevant_levels = [
    level
    for level in available_levels
    if (
        level in required_levels
        or "atmosphere" in level.lower()
        or "ceiling" in level.lower()
    )
]

zarr_variable_inventory_records = []

for level in relevant_levels:

    level_path = f"{zarr_run_path}/{level}"

    objects = fs.ls(
        level_path,
        detail=True,
    )

    variables = sorted([
        item["name"].rstrip("/").rsplit("/", 1)[-1]
        for item in objects
        if item["type"] == "directory"
    ])

    print(f"\n=== {level} ===")
    print("Variables:", variables)

    for variable in variables:

        zarr_variable_inventory_records.append({
            "level": level,
            "variable": variable,
            "group_path": f"{level_path}/{variable}",
        })

zarr_variable_inventory = pd.DataFrame(
    zarr_variable_inventory_records
)

assert not zarr_variable_inventory.empty

assert not zarr_variable_inventory.duplicated(
    subset=["level", "variable"]
).any()

print("\n=== Zarr Variable Inventory ===")

display(zarr_variable_inventory)

print(
    "\nDiscovered variable-level combinations:",
    len(zarr_variable_inventory),
)

In [ ]:
zarr_variable_metadata = {}
zarr_variable_attributes = {}

zarr_feasibility_records = []

for row in ZARR_VARIABLE_REQUIREMENTS.itertuples(
    index=False
):

    level = row.candidate_level
    variable = row.variable

    array_path = build_zarr_array_path(
        ZARR_PILOT_RUN,
        level,
        variable,
    )

    available = (
        zarr_variable_inventory["level"].eq(level)
        & zarr_variable_inventory["variable"].eq(variable)
    ).any()

    if not available:

        zarr_feasibility_records.append({
            "feature": row.feature,
            "level": level,
            "variable": variable,
            "status": "candidate_path_not_found",
            "array_path": array_path,
        })

        continue

    try:

        metadata = read_zarr_json(
            array_path + "/.zarray"
        )

        attributes = read_zarr_json(
            array_path + "/.zattrs"
        )

    except FileNotFoundError:

        zarr_feasibility_records.append({
            "feature": row.feature,
            "level": level,
            "variable": variable,
            "status": "array_metadata_not_found",
            "array_path": array_path,
        })

        continue

    zarr_variable_metadata[row.feature] = metadata

    zarr_variable_attributes[row.feature] = attributes

    zarr_feasibility_records.append({
        "feature": row.feature,
        "level": level,
        "variable": variable,
        "status": "metadata_available",
        "array_path": array_path,
        "shape": tuple(metadata["shape"]),
        "chunks": tuple(metadata["chunks"]),
        "dtype": metadata["dtype"],
        "dimensions": attributes.get("_ARRAY_DIMENSIONS"),
        "units": attributes.get("units"),
        "fill_value": metadata.get("fill_value"),
        "compressor": metadata.get("compressor"),
        "filters": metadata.get("filters"),
        "same_shape_as_temperature": (
            metadata["shape"]
            == temperature_metadata["shape"]
        ),
        "same_chunks_as_temperature": (
            metadata["chunks"]
            == temperature_metadata["chunks"]
        ),
    })

zarr_feasibility_summary = pd.DataFrame(
    zarr_feasibility_records
)

assert len(zarr_feasibility_summary) == 9

print("=== Multi-Variable Zarr Feasibility ===")

display(
    zarr_feasibility_summary[[
        "feature",
        "level",
        "variable",
        "status",
        "shape",
        "chunks",
        "dtype",
        "units",
        "fill_value",
        "same_shape_as_temperature",
        "same_chunks_as_temperature",
    ]]
)

available_count = (
    zarr_feasibility_summary["status"]
    .eq("metadata_available")
    .sum()
)

print(
    "\nVariables with accessible metadata:",
    available_count,
    "/ 9",
)

print("\n=== Metadata Status Counts ===")

display(
    zarr_feasibility_summary["status"].value_counts()
)

In [ ]:
for feature, attributes in (
    zarr_variable_attributes.items()
):

    print(f"\n=== {feature} ===")

    print(
        json.dumps(
            attributes,
            indent=2,
            default=str,
        )
    )

#### 5.7.2 Metadata and Temporal Validation

Validate the storage metadata and coordinate semantics
of each discovered HRRR Zarr weather variable.

Compare forecast initialization, forecast periods,
valid timestamps, and spatial grid coordinates against
the validated temperature reference.

Determine whether the variables can share a common
chunk extraction implementation.

Inspect units and missing-value definitions before
numerical reconciliation with the GRIB reference.

The hourly precipitation accumulation interval must
be independently validated against the corresponding
GRIB records.

In [ ]:
zarr_layout_records = []

for feature, metadata in zarr_variable_metadata.items():

    attributes = zarr_variable_attributes[feature]

    shape = tuple(metadata["shape"])
    chunks = tuple(metadata["chunks"])
    dtype = np.dtype(metadata["dtype"])

    dimensions = attributes.get("_ARRAY_DIMENSIONS")

    same_dimensions = (
        dimensions == array_dims
    )

    same_shape = (
        shape == array_shape
    )

    same_chunks = (
        chunks == chunk_shape
    )

    supported_filters = (
        metadata.get("filters") in (None, [])
    )

    supported_compressor = (
        metadata.get("compressor") is not None
    )

    supported_dtype = (
        np.issubdtype(dtype, np.floating)
        and dtype.itemsize in (2, 4)
    )

    layout_compatible = all([
        metadata["zarr_format"] == 2,
        same_dimensions,
        same_shape,
        same_chunks,
        supported_filters,
        supported_compressor,
        supported_dtype,
        metadata["order"] in ("C", "F"),
        metadata.get(
            "dimension_separator", "."
        ) in (".", "/"),
    ])

    zarr_layout_records.append({
        "feature": feature,
        "dtype": str(dtype),
        "shape": shape,
        "chunks": chunks,
        "dimensions_match": same_dimensions,
        "shape_match": same_shape,
        "chunks_match": same_chunks,
        "filters_supported": supported_filters,
        "fill_value": metadata.get("fill_value"),
        "layout_compatible": layout_compatible,
    })

zarr_layout_validation = pd.DataFrame(
    zarr_layout_records
)

print("=== Zarr Variable Layout Validation ===")

display(zarr_layout_validation)

print(
    "\nCompatible variable layouts:",
    zarr_layout_validation[
        "layout_compatible"
    ].sum(),
    "/",
    len(ZARR_VARIABLE_REQUIREMENTS),
)

In [ ]:
zarr_coordinate_records = []
zarr_variable_lead_mappings = {}

for row in zarr_feasibility_summary.itertuples(
    index=False
):

    feature = row.feature

    if row.status != "metadata_available":

        zarr_coordinate_records.append({
            "feature": feature,
            "status": "metadata_unavailable",
        })

        continue

    group_path = build_zarr_group_path(
        ZARR_PILOT_RUN,
        row.level,
        row.variable,
    )

    print(f"Validating {feature}...")

    ds = xr.open_zarr(
        fs.get_mapper(group_path),
        consolidated=False,
        chunks=None,
        decode_times=True,
        decode_timedelta=True,
    )

    try:

        metadata = zarr_variable_metadata[feature]
        attributes = zarr_variable_attributes[feature]

        # Validate required coordinates.

        required_coordinates = [
            "forecast_reference_time",
            "forecast_period",
            "time",
            "projection_x_coordinate",
            "projection_y_coordinate",
        ]

        assert all(
            name in ds.variables
            for name in required_coordinates
        ), "Required coordinates are missing."

        # Extract temporal coordinates.

        variable_reference_times = pd.to_datetime(
            np.asarray(
                ds["forecast_reference_time"].values
            ).reshape(-1),
            utc=True,
        )

        assert np.issubdtype(
            ds["forecast_period"].dtype,
            np.timedelta64,
        )

        variable_periods = pd.to_timedelta(
            ds["forecast_period"].values
        )

        variable_valid_times = pd.to_datetime(
            ds["time"].values,
            utc=True,
        )

        assert not variable_reference_times.isna().any()
        assert not variable_periods.isna().any()
        assert not variable_valid_times.isna().any()

        assert len(variable_reference_times) == 1

        reference_match = (
            variable_reference_times[0]
            == ZARR_PILOT_RUN
        )

        period_alignment = (
            len(variable_valid_times)
            == len(variable_periods)
            and (
                variable_valid_times
                == variable_reference_times[0]
                + variable_periods
            ).all()
        )

        # Derive forecast lead mapping.

        variable_leads_float = np.asarray(
            variable_periods
            / pd.Timedelta(hours=1)
        )

        assert np.isfinite(
            variable_leads_float
        ).all()

        assert np.equal(
            variable_leads_float,
            np.floor(variable_leads_float),
        ).all()

        variable_leads = (
            variable_leads_float.astype(int)
        )

        assert len(np.unique(variable_leads)) == len(
            variable_leads
        )

        variable_lead_mapping = {
            int(lead): int(index)
            for index, lead in enumerate(variable_leads)
        }

        pilot_leads_available = all(
            lead in variable_lead_mapping
            for lead in ZARR_PILOT_LEADS
        )

        same_forecast_leads = np.array_equal(
            variable_leads,
            lead_hours,
        )

        same_valid_times = (
            variable_valid_times.equals(valid_times)
        )

        # Validate spatial coordinates.

        variable_x = np.asarray(
            ds["projection_x_coordinate"].values
        )

        variable_y = np.asarray(
            ds["projection_y_coordinate"].values
        )

        same_x_grid = np.array_equal(
            variable_x,
            x_coords,
        )

        same_y_grid = np.array_equal(
            variable_y,
            y_coords,
        )

        # Reconcile dimensions against array metadata.

        dimensions = attributes["_ARRAY_DIMENSIONS"]

        array_dimension_match = all(
            ds.sizes.get(dim) == size
            for dim, size in zip(
                dimensions,
                metadata["shape"],
                strict=True,
            )
        )

        shared_coordinates = all([
            reference_match,
            period_alignment,
            pilot_leads_available,
            same_forecast_leads,
            same_valid_times,
            same_x_grid,
            same_y_grid,
            array_dimension_match,
        ])

        zarr_variable_lead_mappings[feature] = (
            variable_lead_mapping
        )

        zarr_coordinate_records.append({
            "feature": feature,
            "forecast_count": len(variable_leads),
            "min_lead": int(variable_leads.min()),
            "max_lead": int(variable_leads.max()),
            "initialization_match": reference_match,
            "period_alignment": period_alignment,
            "pilot_leads_available": pilot_leads_available,
            "same_forecast_leads": same_forecast_leads,
            "same_valid_times": same_valid_times,
            "same_x_grid": same_x_grid,
            "same_y_grid": same_y_grid,
            "array_dimension_match": array_dimension_match,
            "shared_coordinates": shared_coordinates,
            "status": (
                "pass"
                if shared_coordinates
                else "coordinate_mismatch"
            ),
        })

    except (AssertionError, ValueError, KeyError) as exc:

        zarr_coordinate_records.append({
            "feature": feature,
            "status": "validation_failed",
            "error": str(exc),
        })

    finally:

        ds.close()

zarr_coordinate_validation = pd.DataFrame(
    zarr_coordinate_records
)

print("\n=== Variable Coordinate Validation ===")

display(zarr_coordinate_validation)

print(
    "\nVariables sharing temperature coordinates:",
    zarr_coordinate_validation["status"].eq("pass").sum(),
    "/",
    len(ZARR_VARIABLE_REQUIREMENTS),
)

In [ ]:
unit_validation_records = []

for feature, attributes in (
    zarr_variable_attributes.items()
):

    grib_metadata = EXPECTED_METADATA[feature]

    zarr_units = attributes.get("units")

    grib_units = grib_metadata.get("units")

    if zarr_units is None:

        unit_status = "zarr_units_missing"

    elif zarr_units == grib_units:

        unit_status = "exact_unit_match"

    else:

        unit_status = "unit_notation_requires_review"

    unit_validation_records.append({
        "feature": feature,
        "zarr_units": zarr_units,
        "grib_units": grib_units,
        "grib_short_name": grib_metadata.get(
            "short_name"
        ),
        "grib_type_of_level": grib_metadata.get(
            "type_of_level"
        ),
        "grib_step_type": grib_metadata.get(
            "step_type"
        ),
        "zarr_grib_parameter": attributes.get(
            "GRIB_PARAM"
        ),
        "zarr_long_name": attributes.get(
            "long_name"
        ),
        "unit_status": unit_status,
    })

zarr_unit_validation = pd.DataFrame(
    unit_validation_records
)

print("=== Weather Variable Unit Validation ===")

display(zarr_unit_validation)

print("\n=== Unit Status Counts ===")

display(
    zarr_unit_validation["unit_status"].value_counts()
)

In [ ]:
zarr_compatibility_summary = (
    zarr_feasibility_summary[[
        "feature",
        "level",
        "variable",
        "status",
    ]]
    .rename(columns={
        "status": "metadata_status",
    })
    .merge(
        zarr_layout_validation[[
            "feature",
            "layout_compatible",
        ]],
        on="feature",
        how="left",
        validate="one_to_one",
    )
    .merge(
        zarr_coordinate_validation[[
            "feature",
            "status",
        ]].rename(columns={
            "status": "coordinate_status",
        }),
        on="feature",
        how="left",
        validate="one_to_one",
    )
    .merge(
        zarr_unit_validation[[
            "feature",
            "zarr_units",
            "grib_units",
            "unit_status",
        ]],
        on="feature",
        how="left",
        validate="one_to_one",
    )
)

zarr_compatibility_summary[
    "ready_for_extraction_pilot"
] = (
    zarr_compatibility_summary[
        "metadata_status"
    ].eq("metadata_available")
    & zarr_compatibility_summary[
        "layout_compatible"
    ].eq(True)
    & zarr_compatibility_summary[
        "coordinate_status"
    ].eq("pass")
)

print("=== Consolidated Zarr Compatibility Summary ===")

display(zarr_compatibility_summary)

print(
    "\nVariables ready for extraction pilot:",
    zarr_compatibility_summary[
        "ready_for_extraction_pilot"
    ].sum(),
    "/",
    len(ZARR_VARIABLE_REQUIREMENTS),
)

In [ ]:
print("=== Multi-Variable Readiness Check ===")

assert len(zarr_variable_metadata) == 9
assert len(zarr_variable_attributes) == 9

assert zarr_layout_validation[
    "layout_compatible"
].all()

assert zarr_coordinate_validation[
    "status"
].eq("pass").all()

assert len(zarr_compatibility_summary) == 9

assert zarr_compatibility_summary[
    "ready_for_extraction_pilot"
].all()

print("Metadata availability: 9/9")
print("Storage layout compatibility: 9/9")
print("Temporal and spatial compatibility: 9/9")
print("\nMulti-variable extraction readiness: PASS")

#### 5.7.3 Multi-Variable Chunk Extraction

Implement a reusable Zarr extraction function for
all validated HRRR weather variables.

Reuse the established temporal indices and airport
grid mappings to retrieve only the required spatial
chunks.

Support metadata-driven decoding, floating-point
precision, explicit missing-value handling, and
download auditing.

Reuse the previously extracted temperature records
to avoid unnecessary network requests.

Preserve raw weather values and missing-value
provenance for subsequent GRIB reconciliation.

In [ ]:
def extract_zarr_variable(feature):

    # 1. Retrieve validated variable configuration.

    config = zarr_compatibility_summary.loc[
        zarr_compatibility_summary["feature"].eq(feature)
    ]

    assert len(config) == 1

    config = config.iloc[0]

    assert bool(config["ready_for_extraction_pilot"])

    metadata = zarr_variable_metadata[feature]
    attributes = zarr_variable_attributes[feature]

    variable_lead_mapping = (
        zarr_variable_lead_mappings[feature]
    )

    array_path = build_zarr_array_path(
        ZARR_PILOT_RUN,
        config["level"],
        config["variable"],
    )

    # 2. Initialize metadata-driven storage settings.

    dtype = np.dtype(metadata["dtype"])

    chunks = tuple(metadata["chunks"])

    dimensions = attributes["_ARRAY_DIMENSIONS"]

    order = metadata["order"]

    separator = metadata.get(
        "dimension_separator", "."
    )

    fill_value = metadata.get("fill_value")

    if fill_value is not None:
        fill_value = float(fill_value)

    assert metadata["zarr_format"] == 2

    assert dimensions == [
        "time",
        "projection_y_coordinate",
        "projection_x_coordinate",
    ]

    assert chunks == (48, 150, 150)

    assert metadata.get("filters") in (None, [])

    assert order in ("C", "F")

    assert separator in (".", "/")

    assert np.issubdtype(dtype, np.floating)

    # Reject unsupported scaling transformations.

    assert attributes.get("scale_factor", 1) == 1
    assert attributes.get("add_offset", 0) == 0

    codec = numcodecs.get_codec(
        metadata["compressor"]
    )

    expected_bytes = (
        int(np.prod(chunks)) * dtype.itemsize
    )

    time_chunk_size, y_chunk_size, x_chunk_size = chunks

    value_records = []
    download_records = []

    decoded_chunks = {}

    # 3. Download and extract the required chunks.

    for airport_row in airport_zarr_grid.itertuples(
        index=False
    ):

        airport = airport_row.airport

        y_chunk = airport_row.y_index // y_chunk_size
        x_chunk = airport_row.x_index // x_chunk_size

        local_y = airport_row.y_index % y_chunk_size
        local_x = airport_row.x_index % x_chunk_size

        for lead in ZARR_PILOT_LEADS:

            array_index = variable_lead_mapping[lead]

            time_chunk = array_index // time_chunk_size

            local_time = array_index % time_chunk_size

            chunk_key = separator.join(
                map(str, (
                    time_chunk,
                    y_chunk,
                    x_chunk,
                ))
            )

            object_path = f"{array_path}/{chunk_key}"

            # Download each unique chunk only once.

            if chunk_key not in decoded_chunks:

                download_start = perf_counter()

                try:

                    compressed = fs.cat_file(
                        object_path
                    )

                except FileNotFoundError as exc:

                    raise RuntimeError(
                        f"Missing Zarr chunk: {object_path}"
                    ) from exc

                download_seconds = (
                    perf_counter() - download_start
                )

                assert len(compressed) > 0

                # Decode compressed weather data.

                decode_start = perf_counter()

                decoded = codec.decode(compressed)

                array = np.frombuffer(
                    decoded,
                    dtype=dtype,
                )

                assert array.nbytes == expected_bytes, (
                    f"Unexpected decoded size: "
                    f"{feature}/{chunk_key}"
                )

                array = array.reshape(
                    chunks,
                    order=order,
                )

                decode_seconds = (
                    perf_counter() - decode_start
                )

                decoded_chunks[chunk_key] = array

                download_records.append({
                    "feature": feature,
                    "airport": airport,
                    "chunk_key": chunk_key,
                    "object_path": object_path,
                    "compressed_bytes": len(compressed),
                    "decoded_bytes": array.nbytes,
                    "download_seconds": download_seconds,
                    "decode_seconds": decode_seconds,
                    "storage_dtype": str(dtype),
                })

            # 4. Extract the airport-level forecast.

            raw_value = float(
                decoded_chunks[chunk_key][
                    local_time,
                    local_y,
                    local_x,
                ]
            )

            # 5. Preserve missing-value provenance.

            if (
                fill_value is not None
                and np.isfinite(fill_value)
                and raw_value == fill_value
            ):

                value = np.nan
                value_status = "fill_value"

            elif not np.isfinite(raw_value):

                value = np.nan
                value_status = "nonfinite"

            else:

                value = raw_value
                value_status = "valid"

            value_records.append({
                "Origin": airport,
                "feature": feature,
                "value": value,
                "raw_value": raw_value,
                "value_status": value_status,
                "forecast_run_at_utc": ZARR_PILOT_RUN,
                "forecast_valid_at_utc": (
                    ZARR_PILOT_RUN
                    + pd.Timedelta(hours=lead)
                ),
                "forecast_hour": lead,
                "grid_latitude": airport_row.grid_latitude,
                "grid_longitude": airport_row.grid_longitude,
                "chunk_key": chunk_key,
                "array_index": array_index,
                "storage_dtype": str(dtype),
            })

    # 6. Validate extraction completeness.

    values = pd.DataFrame(value_records)

    audit = pd.DataFrame(download_records)

    expected_records = (
        len(ORIGIN_AIRPORTS)
        * len(ZARR_PILOT_LEADS)
    )

    assert len(values) == expected_records

    assert not values.duplicated(
        subset=["Origin", "forecast_hour"]
    ).any()

    assert set(values["Origin"]) == set(
        ORIGIN_AIRPORTS
    )

    assert set(values["forecast_hour"]) == set(
        ZARR_PILOT_LEADS
    )

    assert len(audit) == len(decoded_chunks)

    return values, audit

In [ ]:
temperature_reused = (
    zarr_temperature_long.copy()
)

temperature_reused["raw_value"] = (
    temperature_reused["value"]
)

temperature_reused["value_status"] = "valid"

temperature_reused["storage_dtype"] = str(
    np.dtype(temperature_metadata["dtype"])
)

temperature_reused = temperature_reused.merge(
    airport_zarr_grid[[
        "airport",
        "grid_latitude",
        "grid_longitude",
    ]].rename(columns={
        "airport": "Origin",
    }),
    on="Origin",
    how="left",
    validate="many_to_one",
)

temperature_audit_reused = (
    temperature_download_audit.copy()
)

temperature_audit_reused["feature"] = (
    "temperature_2m"
)

temperature_audit_reused["storage_dtype"] = (
    temperature_reused["storage_dtype"].iloc[0]
)

temperature_audit_reused["object_path"] = (
    temperature_path
    + "/"
    + temperature_audit_reused["chunk_key"]
)

temperature_audit_reused["acquisition_source"] = (
    "reused_temperature_pilot"
)

zarr_extraction_results = {
    "temperature_2m": temperature_reused,
}

zarr_extraction_audits = {
    "temperature_2m": temperature_audit_reused,
}

print("=== Existing Temperature Results ===")

print(
    "Reused temperature records:",
    len(temperature_reused),
)

print(
    "Reused chunk audit records:",
    len(temperature_audit_reused),
)

print("Additional network downloads: 0")

print("\nTemperature reuse: PASS")

In [ ]:
zarr_extraction_failures = []

multi_variable_start = perf_counter()

for feature in ZARR_VARIABLE_REQUIREMENTS["feature"]:

    if feature in zarr_extraction_results:

        print(f"Reusing {feature}...")

        continue

    print(f"\nExtracting {feature}...")

    feature_start = perf_counter()

    try:

        values, audit = extract_zarr_variable(
            feature
        )

        audit["acquisition_source"] = (
            "multi_variable_pilot"
        )

        zarr_extraction_results[feature] = values

        zarr_extraction_audits[feature] = audit

        elapsed = (
            perf_counter() - feature_start
        )

        print("  Records:", len(values))

        print(
            "  Valid:",
            values["value_status"].eq("valid").sum(),
        )

        print(
            "  Missing:",
            values["value"].isna().sum(),
        )

        print("  Chunks:", len(audit))

        print(
            "  Payload:",
            round(
                audit["compressed_bytes"].sum()
                / 1024**2,
                4,
            ),
            "MiB",
        )

        print(
            f"  Processing time: {elapsed:.3f}s"
        )

    except Exception as exc:

        zarr_extraction_failures.append({
            "feature": feature,
            "error_type": type(exc).__name__,
            "error": str(exc),
        })

        print(f"  FAILED: {exc}")

multi_variable_elapsed = (
    perf_counter() - multi_variable_start
)

zarr_failure_audit = pd.DataFrame(
    zarr_extraction_failures,
    columns=["feature", "error_type", "error"],
)

print("\n=== Multi-Variable Extraction Status ===")

print(
    "Successful variables:",
    len(zarr_extraction_results),
    "/ 9",
)

print(
    "Failed variables:",
    len(zarr_failure_audit),
)

print(
    "Current execution time:",
    round(multi_variable_elapsed, 3),
    "seconds",
)

if not zarr_failure_audit.empty:

    display(zarr_failure_audit)

In [ ]:
zarr_weather_long = pd.concat(
    zarr_extraction_results.values(),
    ignore_index=True,
)

zarr_weather_download_audit = pd.concat(
    zarr_extraction_audits.values(),
    ignore_index=True,
)

zarr_weather_long["is_valid"] = (
    zarr_weather_long["value_status"].eq("valid")
)

zarr_weather_long["is_fill_value"] = (
    zarr_weather_long["value_status"].eq("fill_value")
)

zarr_weather_long["is_nonfinite"] = (
    zarr_weather_long["value_status"].eq("nonfinite")
)

zarr_weather_long["is_missing"] = (
    zarr_weather_long["value"].isna()
)

assert not zarr_weather_long.duplicated(
    subset=[
        "Origin",
        "feature",
        "forecast_run_at_utc",
        "forecast_valid_at_utc",
        "forecast_hour",
    ]
).any()

zarr_weather_quality_summary = (
    zarr_weather_long.groupby("feature")
    .agg(
        records=("value", "size"),
        valid_records=("is_valid", "sum"),
        missing_records=("is_missing", "sum"),
        fill_values=("is_fill_value", "sum"),
        nonfinite_values=("is_nonfinite", "sum"),
        min_value=("value", "min"),
        max_value=("value", "max"),
        mean_value=("value", "mean"),
    )
    .reindex(
        ZARR_VARIABLE_REQUIREMENTS["feature"]
    )
)

zarr_weather_quality_summary["coverage_pct"] = (
    zarr_weather_quality_summary["valid_records"]
    / zarr_weather_quality_summary["records"]
    * 100
)

print("=== Multi-Variable Weather Quality Summary ===")

display(
    zarr_weather_quality_summary.round(4)
)

print("\nTotal raw weather records:", len(zarr_weather_long))

print(
    "Total missing values:",
    zarr_weather_long["is_missing"].sum(),
)

print(
    "Total successful variables:",
    zarr_weather_long["feature"].nunique(),
)

In [ ]:
zarr_multi_variable_benchmark = (
    zarr_weather_download_audit.groupby(
        "feature"
    )
    .agg(
        chunks=("chunk_key", "size"),
        compressed_bytes=("compressed_bytes", "sum"),
        decoded_bytes=("decoded_bytes", "sum"),
        download_seconds=("download_seconds", "sum"),
        decode_seconds=("decode_seconds", "sum"),
    )
)

zarr_multi_variable_benchmark["payload_mib"] = (
    zarr_multi_variable_benchmark["compressed_bytes"]
    / 1024**2
)

zarr_multi_variable_benchmark["decoded_mib"] = (
    zarr_multi_variable_benchmark["decoded_bytes"]
    / 1024**2
)

zarr_multi_variable_benchmark[
    "payload_processing_seconds"
] = (
    zarr_multi_variable_benchmark["download_seconds"]
    + zarr_multi_variable_benchmark["decode_seconds"]
)

zarr_multi_variable_benchmark["compression_ratio"] = (
    zarr_multi_variable_benchmark["decoded_bytes"]
    / zarr_multi_variable_benchmark["compressed_bytes"]
)

new_download_audit = (
    zarr_weather_download_audit.loc[
        zarr_weather_download_audit[
            "acquisition_source"
        ].eq("multi_variable_pilot")
    ]
)

print("=== Multi-Variable Acquisition Benchmark ===")

display(
    zarr_multi_variable_benchmark[[
        "chunks",
        "payload_mib",
        "decoded_mib",
        "download_seconds",
        "decode_seconds",
        "payload_processing_seconds",
        "compression_ratio",
    ]].round(4)
)

print(
    "\nTotal successful chunk objects:",
    len(zarr_weather_download_audit),
)

print(
    "New successful chunk downloads:",
    len(new_download_audit),
)

print(
    "New successful payload:",
    round(
        new_download_audit["compressed_bytes"].sum()
        / 1024**2,
        4,
    ),
    "MiB",
)

print(
    "Combined weather payload:",
    round(
        zarr_weather_download_audit[
            "compressed_bytes"
        ].sum() / 1024**2,
        4,
    ),
    "MiB",
)

print(
    "Combined payload processing time:",
    round(
        zarr_multi_variable_benchmark[
            "payload_processing_seconds"
        ].sum(),
        4,
    ),
    "seconds",
)

In [ ]:
EXPECTED_RAW_RECORDS = 9 * 6 * 6

expected_features = set(
    ZARR_VARIABLE_REQUIREMENTS["feature"]
)

actual_features = set(
    zarr_weather_long["feature"]
)

# Core model inputs exclude cloud ceiling.

core_weather = zarr_weather_long.loc[
    zarr_weather_long["feature"].ne(
        "cloud_ceiling_raw"
    )
].copy()

cloud_ceiling_weather = zarr_weather_long.loc[
    zarr_weather_long["feature"].eq(
        "cloud_ceiling_raw"
    )
].copy()

print("=== Multi-Variable Extraction Readiness ===")

print(
    "Raw records:",
    len(zarr_weather_long),
    "/",
    EXPECTED_RAW_RECORDS,
)

print(
    "Core raw weather records:",
    len(core_weather),
    "/ 288",
)

print(
    "Core missing values:",
    core_weather["value"].isna().sum(),
)

print(
    "Cloud ceiling missing values:",
    cloud_ceiling_weather["value"].isna().sum(),
)

print(
    "Missing features:",
    expected_features - actual_features,
)

assert zarr_failure_audit.empty

assert actual_features == expected_features

assert len(zarr_weather_long) == EXPECTED_RAW_RECORDS

assert zarr_weather_long.groupby(
    "feature"
).size().eq(36).all()

assert len(core_weather) == 288

assert len(cloud_ceiling_weather) == 36

print("\nRecord completeness: PASS")

if core_weather["value"].notna().all():

    print("Core weather value completeness: PASS")

else:

    print("Core weather value completeness: REVIEW")

print(
    "Cloud ceiling missingness requires "
    "GRIB reconciliation."
)

print(
    "\nExtraction completed. Numerical "
    "reconciliation is still required."
)

#### 5.7.4 Multi-Variable GRIB Reconciliation

Validate the multi-variable Zarr extraction results
against previously cached GRIB reference records.

Perform one-to-one reconciliation of forecast identity,
spatial grid locations, missing-value masks, and
precision-adjusted numerical values.

Evaluate all nine raw weather variables separately
to identify variable-specific discrepancies.

Verify hourly precipitation against GRIB records
with explicit one-hour accumulation intervals.

Retain unsupported or incompatible variables for
GRIB fallback rather than silently accepting
numerical inconsistencies.

In [ ]:
grib_weather_parts = []
grib_multi_reference_records = []

expected_features = set(
    ZARR_VARIABLE_REQUIREMENTS["feature"]
)

for lead in ZARR_PILOT_LEADS:

    run_at = ZARR_PILOT_RUN

    valid_at = (
        run_at + pd.Timedelta(hours=lead)
    )

    cache_path = (
        WEATHER_CACHE_DIR
        / (
            f"hrrr_{run_at.strftime('%Y%m%d_%H')}"
            f"_f{lead:02d}.parquet"
        )
    )

    assert cache_path.is_file(), (
        f"Missing GRIB cache: {cache_path}"
    )

    frame = pd.read_parquet(cache_path).copy()

    # Validate record completeness.

    assert len(frame) == 9 * 6

    assert set(frame["feature"]) == expected_features

    assert set(frame["Origin"]) == set(
        ORIGIN_AIRPORTS
    )

    assert not frame.duplicated(
        subset=["Origin", "feature"]
    ).any()

    assert frame.groupby(
        "feature"
    ).size().eq(6).all()

    # Normalize timestamps.

    for column in [
        "forecast_run_at_utc",
        "forecast_valid_at_utc",
    ]:

        frame[column] = pd.to_datetime(
            frame[column],
            utc=True,
        )

    assert frame["forecast_run_at_utc"].eq(
        run_at
    ).all()

    assert frame["forecast_valid_at_utc"].eq(
        valid_at
    ).all()

    assert frame["forecast_hour"].eq(lead).all()

    # Verify cached GRIB variable definitions.

    for feature, expected_metadata in (
        EXPECTED_METADATA.items()
    ):

        feature_frame = frame.loc[
            frame["feature"].eq(feature)
        ]

        for key, expected in expected_metadata.items():

            assert feature_frame[key].eq(
                expected
            ).all(), (
                f"GRIB metadata mismatch: "
                f"{feature}, {key}, F{lead}"
            )

    # Verify hourly precipitation interval.

    precipitation = frame.loc[
        frame["feature"].eq("precipitation_1h")
    ]

    assert precipitation["step_type"].eq("accum").all()

    assert precipitation["start_step"].eq(
        lead - 1
    ).all()

    assert precipitation["end_step"].eq(
        lead
    ).all()

    grib_weather_parts.append(frame)

    grib_multi_reference_records.append({
        "forecast_hour": lead,
        "records": len(frame),
        "valid_records": frame["value"].notna().sum(),
        "missing_records": frame["value"].isna().sum(),
        "cache_file": cache_path.name,
        "status": "success",
    })

grib_weather_long = pd.concat(
    grib_weather_parts,
    ignore_index=True,
)

grib_multi_reference_audit = pd.DataFrame(
    grib_multi_reference_records
)

assert len(grib_weather_long) == 324

print("=== GRIB Multi-Variable Reference Audit ===")

display(grib_multi_reference_audit)

print("\nTotal records:", len(grib_weather_long))

print(
    "Total missing values:",
    grib_weather_long["value"].isna().sum(),
)

print("\nGRIB reference validation: PASS")

In [ ]:
FORECAST_MATCH_KEYS = [
    "Origin",
    "feature",
    "forecast_run_at_utc",
    "forecast_valid_at_utc",
    "forecast_hour",
]

grib_multi_compare = (
    grib_weather_long[
        FORECAST_MATCH_KEYS + [
            "value",
            "grid_latitude",
            "grid_longitude",
            "units",
            "start_step",
            "end_step",
        ]
    ]
    .rename(columns={
        "value": "grib_value",
        "grid_latitude": "grib_latitude",
        "grid_longitude": "grib_longitude",
        "units": "grib_units",
    })
    .copy()
)

zarr_multi_compare = (
    zarr_weather_long[
        FORECAST_MATCH_KEYS + [
            "value",
            "raw_value",
            "value_status",
            "grid_latitude",
            "grid_longitude",
            "storage_dtype",
            "chunk_key",
            "array_index",
        ]
    ]
    .rename(columns={
        "value": "zarr_value",
        "grid_latitude": "zarr_latitude",
        "grid_longitude": "zarr_longitude",
    })
    .copy()
)

for frame in [
    grib_multi_compare,
    zarr_multi_compare,
]:

    for column in [
        "forecast_run_at_utc",
        "forecast_valid_at_utc",
    ]:

        frame[column] = pd.to_datetime(
            frame[column],
            utc=True,
        )

multi_variable_comparison = (
    grib_multi_compare.merge(
        zarr_multi_compare,
        on=FORECAST_MATCH_KEYS,
        how="outer",
        validate="one_to_one",
        indicator=True,
    )
)

assert len(multi_variable_comparison) == 324

assert multi_variable_comparison[
    "_merge"
].eq("both").all()

multi_variable_comparison = (
    multi_variable_comparison
    .drop(columns="_merge")
    .sort_values([
        "feature",
        "Origin",
        "forecast_hour",
    ])
    .reset_index(drop=True)
)

print("=== Multi-Variable Forecast Identity ===")

print(
    "Matched records:",
    len(multi_variable_comparison),
    "/ 324",
)

display(
    multi_variable_comparison.groupby(
        "feature"
    ).size().to_frame("matched_records")
)

print("\nForecast identity reconciliation: PASS")

In [ ]:
comparison = multi_variable_comparison

_, _, separation_m = hrrr_geod.inv(
    normalize_longitude(
        comparison["grib_longitude"]
    ).to_numpy(),
    comparison["grib_latitude"].to_numpy(),
    normalize_longitude(
        comparison["zarr_longitude"]
    ).to_numpy(),
    comparison["zarr_latitude"].to_numpy(),
)

comparison["grid_separation_m"] = separation_m

assert np.isfinite(separation_m).all()

comparison["spatial_match"] = (
    comparison["grid_separation_m"]
    <= GRID_MATCH_TOLERANCE_M
)

spatial_validation_summary = (
    comparison.groupby("feature")
    .agg(
        records=("Origin", "size"),
        spatial_matches=("spatial_match", "sum"),
        max_grid_separation_m=(
            "grid_separation_m",
            "max",
        ),
    )
)

print("=== Multi-Variable Spatial Reconciliation ===")

display(spatial_validation_summary)

print(
    "\nSpatial matches:",
    comparison["spatial_match"].sum(),
    "/ 324",
)

assert comparison["spatial_match"].all()

print("\nSpatial reconciliation: PASS")

In [ ]:
comparison["grib_missing"] = (
    comparison["grib_value"].isna()
)

comparison["zarr_missing"] = (
    comparison["zarr_value"].isna()
)

comparison["both_missing"] = (
    comparison["grib_missing"]
    & comparison["zarr_missing"]
)

comparison["both_valid"] = (
    ~comparison["grib_missing"]
    & ~comparison["zarr_missing"]
)

comparison["missing_mask_match"] = (
    comparison["grib_missing"]
    == comparison["zarr_missing"]
)

comparison["missing_mask_mismatch"] = (
    ~comparison["missing_mask_match"]
)

# Reject unexpected infinite values.

for column in ["grib_value", "zarr_value"]:

    assert np.isfinite(
        comparison[column].dropna()
    ).all()

missing_validation_summary = (
    comparison.groupby("feature")
    .agg(
        records=("Origin", "size"),
        grib_missing=("grib_missing", "sum"),
        zarr_missing=("zarr_missing", "sum"),
        both_missing=("both_missing", "sum"),
        both_valid=("both_valid", "sum"),
        mask_matches=("missing_mask_match", "sum"),
        mask_mismatches=("missing_mask_mismatch", "sum"),
    )
)

print("=== Missing-Value Mask Reconciliation ===")

display(missing_validation_summary)

missing_mismatches = comparison.loc[
    comparison["missing_mask_mismatch"]
]

print(
    "\nTotal matching missing records:",
    comparison["both_missing"].sum(),
)

print(
    "Missing-mask mismatches:",
    len(missing_mismatches),
)

if not missing_mismatches.empty:

    display(
        missing_mismatches[[
            "Origin",
            "feature",
            "forecast_hour",
            "grib_value",
            "zarr_value",
            "raw_value",
            "value_status",
        ]]
    )

else:

    print("\nMissing-value mask reconciliation: PASS")

In [ ]:
comparison["expected_zarr_value"] = np.nan
comparison["signed_error"] = np.nan
comparison["absolute_error"] = np.nan
comparison["quantization_error"] = np.nan
comparison["precision_adjusted_error"] = np.nan

comparison["raw_exact_match"] = False
comparison["precision_adjusted_match"] = False

for feature in ZARR_VARIABLE_REQUIREMENTS["feature"]:

    mask = (
        comparison["feature"].eq(feature)
        & comparison["both_valid"]
    )

    if not mask.any():
        continue

    dtype = np.dtype(
        zarr_variable_metadata[feature]["dtype"]
    )

    grib_values = comparison.loc[
        mask, "grib_value"
    ].to_numpy(dtype=np.float64)

    zarr_values = comparison.loc[
        mask, "zarr_value"
    ].to_numpy(dtype=np.float64)

    # Expected values after storage precision conversion.

    expected_values = (
        grib_values
        .astype(dtype)
        .astype(np.float64)
    )

    assert np.isfinite(expected_values).all()

    comparison.loc[
        mask, "expected_zarr_value"
    ] = expected_values

    comparison.loc[
        mask, "signed_error"
    ] = zarr_values - grib_values

    comparison.loc[
        mask, "absolute_error"
    ] = np.abs(zarr_values - grib_values)

    comparison.loc[
        mask, "quantization_error"
    ] = np.abs(expected_values - grib_values)

    comparison.loc[
        mask, "precision_adjusted_error"
    ] = np.abs(zarr_values - expected_values)

    comparison.loc[
        mask, "raw_exact_match"
    ] = zarr_values == grib_values

    comparison.loc[
        mask, "precision_adjusted_match"
    ] = zarr_values == expected_values

print("=== Numerical Comparison Preview ===")

display(
    comparison[[
        "Origin",
        "feature",
        "forecast_hour",
        "grib_value",
        "zarr_value",
        "absolute_error",
        "precision_adjusted_error",
        "precision_adjusted_match",
    ]].head(20)
)

In [ ]:
def calculate_rmse(errors):

    errors = errors.dropna().to_numpy()

    if len(errors) == 0:
        return np.nan

    return float(
        np.sqrt(np.mean(errors ** 2))
    )


multi_variable_validation_summary = (
    comparison.groupby("feature")
    .agg(
        records=("Origin", "size"),
        valid_pairs=("both_valid", "sum"),
        matching_missing=("both_missing", "sum"),
        missing_mismatches=("missing_mask_mismatch", "sum"),
        raw_exact_matches=("raw_exact_match", "sum"),
        precision_adjusted_matches=(
            "precision_adjusted_match",
            "sum",
        ),
        mae=("absolute_error", "mean"),
        rmse=("signed_error", calculate_rmse),
        max_absolute_error=("absolute_error", "max"),
        max_adjusted_error=(
            "precision_adjusted_error",
            "max",
        ),
        max_grid_separation_m=(
            "grid_separation_m",
            "max",
        ),
    )
    .reindex(
        ZARR_VARIABLE_REQUIREMENTS["feature"]
    )
)

summary = multi_variable_validation_summary

summary["numerical_mismatches"] = (
    summary["valid_pairs"]
    - summary["precision_adjusted_matches"]
)

summary["adjusted_match_pct"] = (
    summary["precision_adjusted_matches"]
    / summary["valid_pairs"].replace(0, np.nan)
    * 100
)

summary["reconciliation_status"] = np.select(
    [
        summary["missing_mismatches"].gt(0),
        summary["numerical_mismatches"].gt(0),
        summary["valid_pairs"].eq(0),
    ],
    [
        "missing_mask_mismatch",
        "numerical_mismatch",
        "insufficient_valid_pairs",
    ],
    default="pass",
)

print("=== Multi-Variable Validation Summary ===")

display(summary)

print(
    "\nVariables passing reconciliation:",
    summary["reconciliation_status"].eq("pass").sum(),
    "/ 9",
)

print(
    "Total valid numerical pairs:",
    summary["valid_pairs"].sum(),
)

print(
    "Total precision-adjusted matches:",
    summary["precision_adjusted_matches"].sum(),
)

print(
    "Total matching missing records:",
    summary["matching_missing"].sum(),
)

print(
    "Total numerical mismatches:",
    summary["numerical_mismatches"].sum(),
)

print(
    "Total missing-mask mismatches:",
    summary["missing_mismatches"].sum(),
)

In [ ]:
multi_variable_discrepancies = comparison.loc[
    comparison["missing_mask_mismatch"]
    | (
        comparison["both_valid"]
        & ~comparison["precision_adjusted_match"]
    )
].copy()

print("=== Multi-Variable Discrepancy Audit ===")

print(
    "Discrepancy records:",
    len(multi_variable_discrepancies),
)

if multi_variable_discrepancies.empty:

    print(
        "All paired valid values and missing "
        "masks are consistent."
    )

else:

    display(
        multi_variable_discrepancies[[
            "feature",
            "Origin",
            "forecast_hour",
            "grib_value",
            "zarr_value",
            "raw_value",
            "value_status",
            "storage_dtype",
            "absolute_error",
            "precision_adjusted_error",
            "missing_mask_mismatch",
        ]]
    )

In [ ]:
precipitation_comparison = comparison.loc[
    comparison["feature"].eq(
        "precipitation_1h"
    )
].copy()

assert len(precipitation_comparison) == 36

assert precipitation_comparison["start_step"].eq(
    precipitation_comparison["forecast_hour"] - 1
).all()

assert precipitation_comparison["end_step"].eq(
    precipitation_comparison["forecast_hour"]
).all()

precipitation_comparison["nonzero_reference"] = (
    precipitation_comparison["grib_value"].gt(0)
)

nonzero_precipitation = (
    precipitation_comparison.loc[
        precipitation_comparison["nonzero_reference"]
    ]
)

print("=== Hourly Precipitation Reconciliation ===")

print(
    "Compared records:",
    len(precipitation_comparison),
)

print(
    "Nonzero GRIB precipitation records:",
    len(nonzero_precipitation),
)

print(
    "Maximum GRIB precipitation:",
    precipitation_comparison["grib_value"].max(),
)

print(
    "Maximum Zarr precipitation:",
    precipitation_comparison["zarr_value"].max(),
)

print(
    "Precision-adjusted matches:",
    precipitation_comparison[
        "precision_adjusted_match"
    ].sum(),
    "/ 36",
)

display(
    nonzero_precipitation[[
        "Origin",
        "forecast_hour",
        "start_step",
        "end_step",
        "grib_units",
        "grib_value",
        "zarr_value",
        "absolute_error",
        "precision_adjusted_match",
    ]]
)

if nonzero_precipitation.empty:

    print(
        "\nREVIEW: No nonzero precipitation cases "
        "were available for interval reconciliation."
    )

### 5.8 Historical Archive Precision Validation

Investigate historical changes in HRRR Zarr storage
precision across the training and development periods.

Validate historical data compatibility and quantify
the numerical effects of Float16 storage before
selecting the production weather acquisition backend.

#### 5.8.1 Historical Storage Precision Audit

Inspect variable-level array metadata before and after
the documented archive precision transition.

Determine storage dtype, chunk layout, compression,
fill-value representation, and variable availability.

Do not assume that all variables or forecast cycles
use the same storage precision.

In [ ]:
HISTORICAL_PILOT_RUN = pd.Timestamp(
    "2023-07-14T06:00:00Z"
)

HISTORICAL_AUDIT_RUNS = {
    "training_2023_winter": pd.Timestamp(
        "2023-01-15T06:00:00Z"
    ),
    "training_2023_summer": HISTORICAL_PILOT_RUN,
    "before_transition": pd.Timestamp(
        "2024-06-01T06:00:00Z"
    ),
    "after_transition": pd.Timestamp(
        "2024-06-02T06:00:00Z"
    ),
    "validated_float32_pilot": ZARR_PILOT_RUN,
}

historical_audit_plan = pd.DataFrame([
    {
        "sample": sample,
        "forecast_run_at_utc": run_at,
    }
    for sample, run_at in HISTORICAL_AUDIT_RUNS.items()
])

print("=== Historical Precision Audit Plan ===")

display(historical_audit_plan)

print(
    "Variable-run combinations:",
    len(HISTORICAL_AUDIT_RUNS)
    * len(ZARR_VARIABLE_REQUIREMENTS),
)

In [ ]:
historical_metadata = {}
historical_attributes = {}

historical_metadata_records = []

for sample, run_at in HISTORICAL_AUDIT_RUNS.items():

    print(f"\nAuditing {sample}: {run_at}")

    for row in ZARR_VARIABLE_REQUIREMENTS.itertuples(
        index=False
    ):

        feature = row.feature

        array_path = build_zarr_array_path(
            run_at,
            row.candidate_level,
            row.variable,
        )

        try:

            # Reuse the validated current pilot.

            if run_at == ZARR_PILOT_RUN:

                metadata = zarr_variable_metadata[feature]

                attributes = (
                    zarr_variable_attributes[feature]
                )

            else:

                metadata = read_zarr_json(
                    array_path + "/.zarray"
                )

                attributes = read_zarr_json(
                    array_path + "/.zattrs"
                )

            dtype = np.dtype(metadata["dtype"])

            historical_metadata[
                (sample, feature)
            ] = metadata

            historical_attributes[
                (sample, feature)
            ] = attributes

            historical_metadata_records.append({
                "sample": sample,
                "forecast_run_at_utc": run_at,
                "feature": feature,
                "dtype": str(dtype),
                "dtype_bytes": dtype.itemsize,
                "shape": tuple(metadata["shape"]),
                "chunks": tuple(metadata["chunks"]),
                "fill_value": metadata.get("fill_value"),
                "units": attributes.get("units"),
                "dimensions": attributes.get(
                    "_ARRAY_DIMENSIONS"
                ),
                "compressor": metadata.get("compressor"),
                "status": "metadata_available",
            })

            print(f"  {feature}: {dtype}")

        except FileNotFoundError:

            historical_metadata_records.append({
                "sample": sample,
                "forecast_run_at_utc": run_at,
                "feature": feature,
                "status": "metadata_not_found",
            })

            print(f"  {feature}: metadata not found")

historical_precision_audit = pd.DataFrame(
    historical_metadata_records
)

assert len(historical_precision_audit) == (
    len(HISTORICAL_AUDIT_RUNS)
    * len(ZARR_VARIABLE_REQUIREMENTS)
)

print("\n=== Historical Storage Precision Audit ===")

display(
    historical_precision_audit[[
        "sample",
        "feature",
        "dtype",
        "shape",
        "chunks",
        "fill_value",
        "status",
    ]]
)

In [ ]:
historical_dtype_matrix = (
    historical_precision_audit.pivot(
        index="feature",
        columns="sample",
        values="dtype",
    )
    .reindex(
        index=ZARR_VARIABLE_REQUIREMENTS["feature"],
        columns=list(HISTORICAL_AUDIT_RUNS),
    )
)

print("=== Historical Storage Dtype Matrix ===")

display(historical_dtype_matrix)

print("\n=== Metadata Availability ===")

display(
    historical_precision_audit.groupby(
        ["sample", "status"]
    ).size().unstack(fill_value=0)
)

print("\n=== Storage Precision Counts ===")

display(
    historical_precision_audit.groupby(
        ["sample", "dtype"]
    ).size().unstack(fill_value=0)
)

In [ ]:
historical_fill_records = []

for (sample, feature), metadata in (
    historical_metadata.items()
):

    dtype = np.dtype(metadata["dtype"])

    declared_fill = metadata.get("fill_value")

    if declared_fill is None:

        stored_fill = np.nan
        representation_changed = False

    else:

        declared_fill = float(declared_fill)

        stored_fill = float(
            np.asarray(
                declared_fill,
                dtype=dtype,
            )
        )

        representation_changed = (
            np.isfinite(declared_fill)
            and stored_fill != declared_fill
        )

    historical_fill_records.append({
        "sample": sample,
        "feature": feature,
        "dtype": str(dtype),
        "declared_fill_value": declared_fill,
        "cast_fill_value": stored_fill,
        "representation_changed": (
            representation_changed
        ),
    })

historical_fill_validation = pd.DataFrame(
    historical_fill_records
)

print("=== Historical Fill-Value Validation ===")

display(
    historical_fill_validation.groupby([
        "dtype",
        "declared_fill_value",
        "cast_fill_value",
        "representation_changed",
    ], dropna=False)
    .size()
    .reset_index(name="variable_run_count")
)

print("\n=== Changed Fill-Value Representations ===")

display(
    historical_fill_validation.loc[
        historical_fill_validation[
            "representation_changed"
        ]
    ]
)

#### 5.8.2 Historical GRIB–Zarr Reconciliation

Evaluate the historical Float16 HRRR Zarr archive
using a representative 2023 forecast initialization.

Compare airport-level forecasts with the corresponding
GRIB reference records under the same temporal and
spatial extraction contract.

Quantify storage-induced numerical errors, identify
unexpected discrepancies, and verify missing-value
handling using historical metadata.

Assess extraction correctness separately from the
potential impact of reduced numerical precision
on downstream machine learning features.

In [ ]:
HISTORICAL_SAMPLE = "training_2023_summer"

historical_run = HISTORICAL_AUDIT_RUNS[
    HISTORICAL_SAMPLE
]

historical_lead_mappings = {}
historical_coordinate_records = []

for row in ZARR_VARIABLE_REQUIREMENTS.itertuples(
    index=False
):

    feature = row.feature

    metadata = historical_metadata[
        (HISTORICAL_SAMPLE, feature)
    ]

    attributes = historical_attributes[
        (HISTORICAL_SAMPLE, feature)
    ]

    group_path = build_zarr_group_path(
        historical_run,
        row.candidate_level,
        row.variable,
    )

    print(f"Validating historical {feature}...")

    ds = xr.open_zarr(
        fs.get_mapper(group_path),
        consolidated=False,
        chunks=None,
        decode_times=True,
        decode_timedelta=True,
    )

    try:

        reference_times = pd.to_datetime(
            np.asarray(
                ds["forecast_reference_time"].values
            ).reshape(-1),
            utc=True,
        )

        periods = pd.to_timedelta(
            ds["forecast_period"].values
        )

        times = pd.to_datetime(
            ds["time"].values,
            utc=True,
        )

        assert len(reference_times) == 1

        assert reference_times[0] == historical_run

        assert not periods.isna().any()
        assert not times.isna().any()

        assert (
            times == historical_run + periods
        ).all()

        leads_float = np.asarray(
            periods / pd.Timedelta(hours=1)
        )

        assert np.isfinite(leads_float).all()

        assert np.equal(
            leads_float,
            np.floor(leads_float)
        ).all()

        leads = leads_float.astype(int)

        assert np.array_equal(
            leads,
            np.arange(1, 49)
        )

        mapping = {
            int(lead): index
            for index, lead in enumerate(leads)
        }

        assert all(
            lead in mapping
            for lead in ZARR_PILOT_LEADS
        )

        same_x = np.array_equal(
            ds["projection_x_coordinate"].values,
            x_coords,
        )

        same_y = np.array_equal(
            ds["projection_y_coordinate"].values,
            y_coords,
        )

        assert same_x and same_y

        assert attributes["_ARRAY_DIMENSIONS"] == array_dims

        assert tuple(metadata["shape"]) == array_shape

        assert tuple(metadata["chunks"]) == chunk_shape

        assert all(
            ds.sizes.get(dim) == size
            for dim, size in zip(
                attributes["_ARRAY_DIMENSIONS"],
                metadata["shape"],
                strict=True,
            )
        )

        historical_lead_mappings[feature] = mapping

        historical_coordinate_records.append({
            "feature": feature,
            "forecast_count": len(leads),
            "initialization_match": True,
            "same_x_grid": same_x,
            "same_y_grid": same_y,
            "storage_dtype": str(
                np.dtype(metadata["dtype"])
            ),
            "status": "pass",
        })

    finally:

        ds.close()

historical_coordinate_validation = pd.DataFrame(
    historical_coordinate_records
)

print("\n=== Historical Coordinate Validation ===")

display(historical_coordinate_validation)

assert len(historical_lead_mappings) == 9

print("\nHistorical coordinate validation: PASS")

In [ ]:
historical_grib_references = []

for lead in ZARR_PILOT_LEADS:

    valid_at = (
        historical_run
        + pd.Timedelta(hours=lead)
    )

    reference = select_hrrr_forecast(valid_at)

    assert reference["forecast_run_at_utc"] == historical_run

    assert reference["forecast_hour"] == lead

    assert reference["forecast_valid_at_utc"] == valid_at

    assert (
        reference["assumed_available_at_utc"]
        <= reference["prediction_at_utc"]
    )

    historical_grib_references.append(reference)

historical_grib_parts = []
historical_grib_audit_records = []

for reference in historical_grib_references:

    lead = reference["forecast_hour"]

    print(f"Acquiring historical GRIB F{lead:02d}...")

    frame, stats = get_cached_weather(reference)

    assert len(frame) == 54

    assert not frame.duplicated(
        subset=["Origin", "feature"]
    ).any()

    historical_grib_parts.append(frame)

    historical_grib_audit_records.append({
        "forecast_hour": lead,
        "cache_hit": stats["CacheHit"],
        "downloaded_bytes": stats["DownloadedBytes"],
        "seconds": stats["Seconds"],
        "records": len(frame),
    })

    print(
        f"  Cache hit: {stats['CacheHit']} | "
        f"Payload: "
        f"{stats['DownloadedBytes'] / 1024**2:.2f} MiB | "
        f"Time: {stats['Seconds']:.2f}s"
    )

historical_grib_long = pd.concat(
    historical_grib_parts,
    ignore_index=True,
)

historical_grib_audit = pd.DataFrame(
    historical_grib_audit_records
)

assert len(historical_grib_long) == 324

assert not historical_grib_long.duplicated(
    subset=FORECAST_MATCH_KEYS
).any()

print("\n=== Historical GRIB Acquisition Audit ===")

display(historical_grib_audit)

print(
    "Total downloaded payload:",
    historical_grib_audit["downloaded_bytes"].sum()
    / 1024**2,
    "MiB",
)

print("\nHistorical GRIB acquisition: PASS")

In [ ]:
def extract_historical_zarr_variable(feature):

    metadata = historical_metadata[
        (HISTORICAL_SAMPLE, feature)
    ]

    attributes = historical_attributes[
        (HISTORICAL_SAMPLE, feature)
    ]

    config = ZARR_VARIABLE_REQUIREMENTS.loc[
        ZARR_VARIABLE_REQUIREMENTS["feature"].eq(feature)
    ].iloc[0]

    array_path = build_zarr_array_path(
        historical_run,
        config["candidate_level"],
        config["variable"],
    )

    dtype = np.dtype(metadata["dtype"])
    chunks = tuple(metadata["chunks"])
    order = metadata["order"]

    separator = metadata.get(
        "dimension_separator", "."
    )

    assert metadata["zarr_format"] == 2

    assert metadata.get("filters") in (None, [])

    assert attributes["_ARRAY_DIMENSIONS"] == array_dims

    assert attributes.get("scale_factor", 1) == 1
    assert attributes.get("add_offset", 0) == 0

    assert dtype in (
        np.dtype("float16"),
        np.dtype("float32"),
    )

    assert order in ("C", "F")
    assert separator in (".", "/")

    codec = numcodecs.get_codec(
        metadata["compressor"]
    )

    declared_fill = metadata.get("fill_value")

    stored_fill = (
        float(np.asarray(declared_fill, dtype=dtype))
        if declared_fill is not None
        else np.nan
    )

    nt, ny, nx = chunks

    expected_bytes = (
        int(np.prod(chunks)) * dtype.itemsize
    )

    mapping = historical_lead_mappings[feature]

    decoded_chunks = {}
    records = []
    audit_records = []

    for airport in airport_zarr_grid.itertuples(
        index=False
    ):

        cy, ly = divmod(airport.y_index, ny)
        cx, lx = divmod(airport.x_index, nx)

        for lead in ZARR_PILOT_LEADS:

            index = mapping[lead]

            ct, lt = divmod(index, nt)

            chunk_key = separator.join(
                map(str, (ct, cy, cx))
            )

            if chunk_key not in decoded_chunks:

                path = f"{array_path}/{chunk_key}"

                download_start = perf_counter()

                compressed = fs.cat_file(path)

                download_seconds = (
                    perf_counter() - download_start
                )

                assert len(compressed) > 0

                decode_start = perf_counter()

                decoded = codec.decode(compressed)

                array = np.frombuffer(
                    decoded,
                    dtype=dtype,
                )

                assert array.nbytes == expected_bytes

                array = array.reshape(
                    chunks,
                    order=order,
                )

                decode_seconds = (
                    perf_counter() - decode_start
                )

                decoded_chunks[chunk_key] = array

                audit_records.append({
                    "feature": feature,
                    "chunk_key": chunk_key,
                    "compressed_bytes": len(compressed),
                    "download_seconds": download_seconds,
                    "decode_seconds": decode_seconds,
                    "storage_dtype": str(dtype),
                })

            raw_value = float(
                decoded_chunks[chunk_key][lt, ly, lx]
            )

            if (
                np.isfinite(stored_fill)
                and raw_value == stored_fill
            ):

                value = np.nan
                status = "fill_value"

            elif not np.isfinite(raw_value):

                value = np.nan
                status = "nonfinite"

            else:

                value = raw_value
                status = "valid"

            records.append({
                "Origin": airport.airport,
                "feature": feature,
                "value": value,
                "raw_value": raw_value,
                "value_status": status,
                "storage_dtype": str(dtype),
                "stored_fill_value": stored_fill,
                "forecast_run_at_utc": historical_run,
                "forecast_valid_at_utc": (
                    historical_run
                    + pd.Timedelta(hours=lead)
                ),
                "forecast_hour": lead,
                "grid_latitude": airport.grid_latitude,
                "grid_longitude": airport.grid_longitude,
            })

    values = pd.DataFrame(records)
    audit = pd.DataFrame(audit_records)

    assert len(values) == 36

    assert not values.duplicated(
        subset=FORECAST_MATCH_KEYS
    ).any()

    return values, audit

In [ ]:
historical_zarr_results = {}
historical_zarr_audits = {}

In [ ]:
for feature in ZARR_VARIABLE_REQUIREMENTS["feature"]:

    if feature in historical_zarr_results:

        print(f"Reusing {feature}...")
        continue

    print(f"Extracting historical {feature}...")

    values, audit = extract_historical_zarr_variable(
        feature
    )

    historical_zarr_results[feature] = values
    historical_zarr_audits[feature] = audit

    print(
        f"  Records: {len(values)} | "
        f"Missing: {values['value'].isna().sum()} | "
        f"Payload: "
        f"{audit['compressed_bytes'].sum() / 1024**2:.3f} MiB"
    )

historical_zarr_long = pd.concat(
    historical_zarr_results.values(),
    ignore_index=True,
)

historical_zarr_download_audit = pd.concat(
    historical_zarr_audits.values(),
    ignore_index=True,
)

assert len(historical_zarr_long) == 324

assert not historical_zarr_long.duplicated(
    subset=FORECAST_MATCH_KEYS
).any()

print("\n=== Historical Zarr Value Summary ===")

display(
    historical_zarr_long.groupby(
        ["feature", "value_status"]
    ).size().unstack(fill_value=0)
)

print(
    "\nDownloaded payload:",
    historical_zarr_download_audit[
        "compressed_bytes"
    ].sum() / 1024**2,
    "MiB",
)

print("\nHistorical Zarr extraction: PASS")

In [ ]:
historical_comparison = (
    historical_grib_long[
        FORECAST_MATCH_KEYS + [
            "value",
            "grid_latitude",
            "grid_longitude",
        ]
    ]
    .merge(
        historical_zarr_long,
        on=FORECAST_MATCH_KEYS,
        how="outer",
        suffixes=("_grib", "_zarr"),
        validate="one_to_one",
        indicator=True,
    )
)

assert len(historical_comparison) == 324

assert historical_comparison["_merge"].eq("both").all()

hc = historical_comparison

# Spatial reconciliation.

_, _, separation = hrrr_geod.inv(
    normalize_longitude(
        hc["grid_longitude_grib"]
    ).to_numpy(),
    hc["grid_latitude_grib"].to_numpy(),
    normalize_longitude(
        hc["grid_longitude_zarr"]
    ).to_numpy(),
    hc["grid_latitude_zarr"].to_numpy(),
)

hc["grid_separation_m"] = separation

assert np.isfinite(separation).all()

assert (separation <= GRID_MATCH_TOLERANCE_M).all()

# Missing-value reconciliation.

hc["grib_missing"] = hc["value_grib"].isna()
hc["zarr_missing"] = hc["value_zarr"].isna()

hc["both_missing"] = (
    hc["grib_missing"] & hc["zarr_missing"]
)

hc["both_valid"] = (
    ~hc["grib_missing"] & ~hc["zarr_missing"]
)

hc["missing_mismatch"] = (
    hc["grib_missing"] != hc["zarr_missing"]
)

hc["expected_zarr_value"] = np.nan
hc["absolute_error"] = np.nan
hc["quantization_error"] = np.nan
hc["precision_adjusted_error"] = np.nan

hc["raw_exact_match"] = False
hc["precision_adjusted_match"] = False
hc["storage_overflow"] = False
hc["fill_value_collision"] = False

for feature in ZARR_VARIABLE_REQUIREMENTS["feature"]:

    mask = (
        hc["feature"].eq(feature)
        & ~hc["grib_missing"]
    )

    dtype = np.dtype(
        historical_metadata[
            (HISTORICAL_SAMPLE, feature)
        ]["dtype"]
    )

    reference = hc.loc[
        mask, "value_grib"
    ].to_numpy(dtype=np.float64)

    assert np.isfinite(reference).all()

    observed = hc.loc[
        mask, "value_zarr"
    ].to_numpy(dtype=np.float64)

    with np.errstate(over="ignore", invalid="ignore"):

        expected = (
            reference
            .astype(dtype)
            .astype(np.float64)
        )

    overflow = ~np.isfinite(expected)

    fill_collision = (
        np.isfinite(expected)
        & (
            expected == hc.loc[
                mask, "stored_fill_value"
            ].to_numpy()
        )
    )

    comparable = (
        np.isfinite(expected)
        & np.isfinite(observed)
        & ~fill_collision
    )

    hc.loc[mask, "expected_zarr_value"] = expected
    hc.loc[mask, "storage_overflow"] = overflow
    hc.loc[mask, "fill_value_collision"] = fill_collision

    hc.loc[mask, "absolute_error"] = np.abs(
        observed - reference
    )

    hc.loc[mask, "quantization_error"] = np.where(
        np.isfinite(expected),
        np.abs(expected - reference),
        np.nan,
    )

    adjusted_error = np.full(
        len(reference), np.nan
    )

    adjusted_error[comparable] = np.abs(
        observed[comparable] - expected[comparable]
    )

    hc.loc[
        mask, "precision_adjusted_error"
    ] = adjusted_error

    hc.loc[mask, "raw_exact_match"] = (
        observed == reference
    )

    hc.loc[mask, "precision_adjusted_match"] = (
        comparable
        & (observed == expected)
    )

print("=== Historical Numerical Reconciliation ===")

display(hc.head(20))

print(
    "\nMaximum grid separation:",
    hc["grid_separation_m"].max(),
    "m",
)

In [ ]:
historical_validation_summary = (
    hc.groupby("feature")
    .agg(
        records=("Origin", "size"),
        valid_pairs=("both_valid", "sum"),
        matching_missing=("both_missing", "sum"),
        missing_mismatches=("missing_mismatch", "sum"),
        raw_exact_matches=("raw_exact_match", "sum"),
        adjusted_matches=("precision_adjusted_match", "sum"),
        mae=("absolute_error", "mean"),
        max_raw_error=("absolute_error", "max"),
        max_quantization_error=("quantization_error", "max"),
        max_adjusted_error=("precision_adjusted_error", "max"),
        storage_overflows=("storage_overflow", "sum"),
        fill_collisions=("fill_value_collision", "sum"),
    )
    .reindex(
        ZARR_VARIABLE_REQUIREMENTS["feature"]
    )
)

hs = historical_validation_summary

hs["numerical_mismatches"] = (
    hs["valid_pairs"] - hs["adjusted_matches"]
)

hs["adjusted_match_pct"] = (
    hs["adjusted_matches"]
    / hs["valid_pairs"].replace(0, np.nan)
    * 100
)

hs["status"] = np.select(
    [
        hs["storage_overflows"].gt(0),
        hs["fill_collisions"].gt(0),
        hs["missing_mismatches"].gt(0),
        hs["numerical_mismatches"].gt(0),
        hs["valid_pairs"].eq(0),
    ],
    [
        "storage_overflow",
        "fill_collision",
        "missing_mask_mismatch",
        "numerical_mismatch",
        "insufficient_valid_pairs",
    ],
    default="pass",
)

print("=== Historical Float16 Validation Summary ===")

display(hs)

print(
    "\nVariables passing reconciliation:",
    hs["status"].eq("pass").sum(),
    "/ 9",
)

print(
    "Total storage overflows:",
    hs["storage_overflows"].sum(),
)

print(
    "Total fill-value collisions:",
    hs["fill_collisions"].sum(),
)

historical_discrepancies = hc.loc[
    hc["missing_mismatch"]
    | hc["storage_overflow"]
    | hc["fill_value_collision"]
    | (
        hc["both_valid"]
        & ~hc["precision_adjusted_match"]
    )
]

print(
    "\nUnresolved discrepancy records:",
    len(historical_discrepancies),
)

display(
    historical_discrepancies[[
        "Origin",
        "feature",
        "forecast_hour",
        "value_grib",
        "value_zarr",
        "raw_value",
        "expected_zarr_value",
        "precision_adjusted_error",
        "value_status",
        "storage_overflow",
        "fill_value_collision",
    ]]
)

#### 5.8.3 Weather Acquisition Backend Decision

The feasibility experiments support a Zarr-first
weather acquisition architecture with GRIB fallback.

Both historical Float16 and newer Float32 archives
successfully reproduced the corresponding GRIB
reference values after accounting for storage precision.

The acquisition pipeline must dynamically inspect
array metadata rather than assume a fixed storage
dtype or archive transition date.

Preserve backend provenance, storage precision,
forecast identity, and missing-value information
throughout the acquisition process.

Use GRIB fallback when Zarr metadata, chunks,
coordinates, variable definitions, or extracted
values fail validation.

The downstream impact of historical precision
differences will be evaluated separately during
model development.

In [ ]:
feasibility_results = pd.DataFrame([
    {
        "pilot": "2023_float16",
        "forecast_run_at_utc": historical_run,
        "storage_dtype": "float16",
        "variables": len(historical_validation_summary),

        "raw_records": int(
            historical_validation_summary["records"].sum()
        ),

        "valid_pairs": int(
            historical_validation_summary["valid_pairs"].sum()
        ),

        "raw_exact_matches": int(
            historical_validation_summary[
                "raw_exact_matches"
            ].sum()
        ),

        "adjusted_matches": int(
            historical_validation_summary[
                "adjusted_matches"
            ].sum()
        ),

        "matching_missing": int(
            historical_validation_summary[
                "matching_missing"
            ].sum()
        ),

        "missing_mismatches": int(
            historical_validation_summary[
                "missing_mismatches"
            ].sum()
        ),

        "numerical_mismatches": int(
            historical_validation_summary[
                "numerical_mismatches"
            ].sum()
        ),

        "zarr_payload_mib": (
            historical_zarr_download_audit[
                "compressed_bytes"
            ].sum() / 1024**2
        ),

        "zarr_payload_processing_seconds": (
            historical_zarr_download_audit[
                "download_seconds"
            ].sum()
            + historical_zarr_download_audit[
                "decode_seconds"
            ].sum()
        ),
    },
    {
        "pilot": "2024_float32",
        "forecast_run_at_utc": ZARR_PILOT_RUN,
        "storage_dtype": "float32",
        "variables": len(multi_variable_validation_summary),

        "raw_records": int(
            multi_variable_validation_summary[
                "records"
            ].sum()
        ),

        "valid_pairs": int(
            multi_variable_validation_summary[
                "valid_pairs"
            ].sum()
        ),

        "raw_exact_matches": int(
            multi_variable_validation_summary[
                "raw_exact_matches"
            ].sum()
        ),

        "adjusted_matches": int(
            multi_variable_validation_summary[
                "precision_adjusted_matches"
            ].sum()
        ),

        "matching_missing": int(
            multi_variable_validation_summary[
                "matching_missing"
            ].sum()
        ),

        "missing_mismatches": int(
            multi_variable_validation_summary[
                "missing_mismatches"
            ].sum()
        ),

        "numerical_mismatches": int(
            multi_variable_validation_summary[
                "numerical_mismatches"
            ].sum()
        ),

        "zarr_payload_mib": (
            zarr_weather_download_audit[
                "compressed_bytes"
            ].sum() / 1024**2
        ),

        "zarr_payload_processing_seconds": (
            zarr_multi_variable_benchmark[
                "payload_processing_seconds"
            ].sum()
        ),
    },
]).set_index("pilot")

feasibility_results["raw_exact_match_pct"] = (
    feasibility_results["raw_exact_matches"]
    / feasibility_results["valid_pairs"]
    * 100
)

feasibility_results["adjusted_match_pct"] = (
    feasibility_results["adjusted_matches"]
    / feasibility_results["valid_pairs"]
    * 100
)

print("=== Consolidated Weather Feasibility Results ===")

display(feasibility_results.T.round(6))

assert feasibility_results["variables"].eq(9).all()

assert feasibility_results["raw_records"].eq(324).all()

assert feasibility_results[
    "numerical_mismatches"
].eq(0).all()

assert feasibility_results[
    "missing_mismatches"
].eq(0).all()

assert feasibility_results[
    "adjusted_match_pct"
].eq(100).all()

assert historical_validation_summary[
    "storage_overflows"
].eq(0).all()

assert historical_validation_summary[
    "fill_collisions"
].eq(0).all()

print("\nWeather feasibility sample validation: PASS")

In [ ]:
weather_backend_policy = pd.DataFrame([
    {
        "condition": "validated_float32_array",
        "action": "use_zarr",
        "reason": "Validated numerical equivalence",
    },
    {
        "condition": "validated_float16_array",
        "action": "use_zarr_with_precision_tracking",
        "reason": "Validated storage-adjusted equivalence",
    },
    {
        "condition": "missing_zarr_metadata",
        "action": "fallback_to_grib",
        "reason": "Cannot verify storage contract",
    },
    {
        "condition": "unsupported_array_layout",
        "action": "fallback_to_grib",
        "reason": "Unsupported decoding configuration",
    },
    {
        "condition": "coordinate_mismatch",
        "action": "fallback_to_grib",
        "reason": "Incorrect temporal or spatial alignment",
    },
    {
        "condition": "missing_or_corrupt_chunk",
        "action": "retry_then_fallback_to_grib",
        "reason": "Acquisition failure",
    },
    {
        "condition": "unexpected_core_missing_value",
        "action": "fallback_to_grib",
        "reason": "Core weather feature incomplete",
    },
    {
        "condition": "cloud_ceiling_fill_value",
        "action": "preserve_missing",
        "reason": "Expected missingness for optional variable",
    },
    {
        "condition": "unexpected_numerical_discrepancy",
        "action": "fallback_to_grib_and_log",
        "reason": "Failed reference reconciliation",
    },
    {
        "condition": "both_backends_fail",
        "action": "record_failure_without_imputation",
        "reason": "Prevent silent data corruption",
    },
])

print("=== Proposed Weather Backend Policy ===")

display(weather_backend_policy)

print("\nPrimary backend: Zarr")
print("Reference and fallback backend: GRIB")
print("Storage precision: Metadata-driven")
print("Cloud ceiling: Excluded from initial V5 features")

In [ ]:
WEATHER_FEASIBILITY_DIR = (
    ROOT / "artifacts" / "weather_feasibility"
)

WEATHER_FEASIBILITY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

feasibility_tables = {
    "feasibility_summary.csv":
        feasibility_results,

    "temperature_benchmark.csv":
        temperature_benchmark_summary,

    "temperature_validation.csv":
        temperature_validation_summary,

    "multivariable_benchmark_2024.csv":
        zarr_multi_variable_benchmark,

    "multivariable_validation_2024.csv":
        multi_variable_validation_summary,

    "historical_precision_audit.csv":
        historical_precision_audit,

    "historical_validation_2023.csv":
        historical_validation_summary,

    "historical_grib_acquisition.csv":
        historical_grib_audit,

    "historical_zarr_acquisition.csv":
        historical_zarr_download_audit,

    "backend_selection_policy.csv":
        weather_backend_policy,
}

saved_artifacts = []

for filename, frame in feasibility_tables.items():

    path = WEATHER_FEASIBILITY_DIR / filename

    frame.to_csv(
        path,
        index=True,
    )

    assert path.is_file()
    assert path.stat().st_size > 0

    restored = pd.read_csv(path)

    assert len(restored) == len(frame)

    saved_artifacts.append({
        "artifact": filename,
        "rows": len(frame),
        "bytes": path.stat().st_size,
    })

weather_feasibility_artifact_manifest = pd.DataFrame(
    saved_artifacts
)

weather_feasibility_artifact_manifest.to_csv(
    WEATHER_FEASIBILITY_DIR / "manifest.csv",
    index=False,
)

print("=== Weather Feasibility Artifacts ===")

display(weather_feasibility_artifact_manifest)

print("\nOutput directory:", WEATHER_FEASIBILITY_DIR)

print(
    "Saved experiment tables:",
    len(saved_artifacts),
)

print("\nArtifact persistence: PASS")

## 6. Conclusion

The weather feasibility study demonstrated that
historical HRRR forecasts can be acquired and aligned
with the 24-hour-ahead flight delay prediction task
under the defined forecast availability assumptions.

Key findings:

- Validated forecast initialization, lead time,
  valid timestamps, and airport-level spatial mapping.

- Reconciled GRIB and Zarr forecasts across nine
  raw weather variables and six origin airports.

- Compared 648 weather records across two historical
  forecast cycles, including 600 valid numerical
  pairs and 48 matching missing-value records.

- Achieved 100% precision-adjusted numerical
  agreement in both Float16 and Float32 pilots.

- Observed a 17.49x payload processing speedup
  in the Float32 temperature benchmark.

- Identified historical storage precision changes
  and variable-dependent missing-value semantics.

- Selected Zarr as the proposed primary acquisition
  backend, with GRIB retained as the reference
  and fallback implementation.

Limitations:

- The validation experiments cover a limited number
  of forecast cycles and do not establish full-year
  archive completeness.

- Forecast availability is based on an assumed
  publication lag rather than independently
  verified historical availability timestamps.

- Some Zarr variables lack explicit unit metadata.

- The benchmark does not establish a general
  end-to-end production speedup.

- The downstream model impact of Float16 storage
  precision remains to be evaluated.

The next phase will implement a reusable weather
acquisition pipeline, persistent caching, quality
validation, and point-in-time feature joins for
weather-enhanced flight delay prediction.